# Amanah AI — Verification Pipeline
**AI-Powered Verification and Correction of Quranic and Prophetic Quotations**

This notebook contains the complete Python verification pipeline of the project: detection, retrieval (RAG-style evidence lookup), word-level alignment, decision and source-backed correction. It never generates scripture: every reference and correction is copied verbatim from the bundled corpora, otherwise the result is *Incorrect* or *Needs Human Review*.

## 1. Configuration
Choose where the corpora come from and whether the optional LLM step is enabled.

In [ ]:
import os, sys, json, pathlib

WORKDIR = (pathlib.Path("/content/amanah_ai") if pathlib.Path("/content").exists() else pathlib.Path("amanah_ai_workdir")).resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
for folder in ("index", "data", "demo"):
    (WORKDIR / folder).mkdir(exist_ok=True)

USE_LLM = False          
print("Working directory:", WORKDIR)

## 2. Provide the corpora
Upload `index/quran.idx.gz` and `index/hadith.idx.gz` from the repository (fast), or `data/quran.json` and `data/hadith.json` (the indexes are then built here). Skipped automatically when the files already exist.

In [ ]:
have_index = all((WORKDIR / "index" / n).exists() for n in ("quran.idx.gz", "hadith.idx.gz"))
have_json = all((WORKDIR / "data" / n).exists() for n in ("quran.json", "hadith.json"))
if not (have_index or have_json):
    try:
        from google.colab import files
        print("Select the two index files (or the two JSON corpora) from the repository.")
        for name, payload in files.upload().items():
            target = "index" if name.endswith(".idx.gz") else "data"
            (WORKDIR / target / name).write_bytes(payload)
    except ImportError:
        raise SystemExit("Place index/*.idx.gz or data/*.json in the working directory, then re-run this cell.")
print({p.name: p.stat().st_size for folder in ("index", "data") for p in (WORKDIR / folder).glob("*")})

### Module: `normalization.py`
Arabic normalisation: phonetic skeletons, diacritic handling and tokenisation used by every other stage.

In [ ]:
%%writefile normalization.py

from __future__ import annotations

import re
import unicodedata
from typing import List, Tuple

_ZERO_WIDTH = re.compile("[\u200b-\u200f\u202a-\u202e\u2066-\u2069\ufeff]")
_MARKS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED\u0640]")
_MATCH_MARKS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0640]")
_PUNCTUATION = re.compile(r"[،؛؟!،.,:;'\"()\[\]{}<>«»\-_/\\|@#$%^&*+=~`﴿﴾]")
_SPACES = re.compile(r"\s+")
_ALEF_FORMS = re.compile(r"[أإآٱ]")
_PERSIAN = str.maketrans({"ی": "ي", "ې": "ي", "ک": "ك", "ە": "ه", "ہ": "ه", "ۀ": "ه"})

STOPWORDS = frozenset(
    """من في على ان أن إن الى إلى عن مع ما لا لم لن قد و ثم أو او هو هي هم انت أنتم كان كانت يكون تكون قال قالت
هذا هذه ذلك تلك الذي التي الذين اللاتي اللائي كل بعض غير عند بين حتى إذا اذا لو لكن بل يا أيها ايها""".split()
)


def _prefold(text: str) -> str:
    text = unicodedata.normalize("NFC", text).replace("ﷲ", "الله")
    return _ZERO_WIDTH.sub("", text).translate(_PERSIAN)


def normalize_strict(text) -> str:
    if not text:
        return ""
    text = _MARKS.sub("", _prefold(text))
    text = _ALEF_FORMS.sub("ا", text)
    return _SPACES.sub(" ", _PUNCTUATION.sub(" ", text)).strip()


def normalize_lenient(text) -> str:
    return normalize_strict(text).replace("ة", "ه").replace("ى", "ي")


def normalize_for_matching(text) -> str:
    if not text:
        return ""
    text = _MATCH_MARKS.sub("", _prefold(text))
    for source, target in (("أ", "ا"), ("إ", "ا"), ("آ", "ا"), ("ٱ", "ا"), ("ؤ", "و"), ("ئ", "ي"), ("ة", "ه"), ("ى", "ي")):
        text = text.replace(source, target)
    text = re.sub(r"[^\u0621-\u064A\s]", " ", text)   
    return _SPACES.sub(" ", text).strip()


def tokenize(text: str) -> List[str]:
    return [token for token in text.split() if token]


def content_words(tokens) -> List[str]:
    """Drop stop-words and single-letter tokens."""
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]


def char_ngrams(text: str, n: int = 4) -> set:
    return {text[i : i + n] for i in range(len(text) - n + 1)}


_ARABIC_LETTER = re.compile(r"[\u0621-\u064A]")


def aligned_words(text: str) -> List[Tuple[str, str]]:
    pairs = []
    for word in text.split():
        skeleton = normalize_for_matching(word)
        if skeleton and _ARABIC_LETTER.search(skeleton):
            pairs.append((word, skeleton.replace(" ", "")))
    return pairs


_VOWELS = {"\u064E": "a", "\u064F": "u", "\u0650": "i", "\u064B": "A", "\u064C": "U", "\u064D": "I", "\u0651": "~"}


def vowel_signature(word: str) -> List[Tuple[str, str]]:
    word = _prefold(word).replace("ٱ", "ا")
    signature: List[Tuple[str, str]] = []
    for char in word:
        if char in _VOWELS:
            if signature:
                letter, marks = signature[-1]
                signature[-1] = (letter, "".join(sorted(set(marks + _VOWELS[char]))))
        elif "\u0621" <= char <= "\u064A":
            signature.append((char, ""))
    return signature


_PHONETIC_CLASSES = {"ص": "س", "ث": "س", "ذ": "ز", "ظ": "ز", "ض": "ز", "ط": "ت", "ك": "ق", "ح": "ه", "غ": "ع"}
_PHONETIC_TABLE = str.maketrans(_PHONETIC_CLASSES)


def phonetic_key(skeleton_word: str) -> str:
    word = skeleton_word.translate(_PHONETIC_TABLE)
    return "".join(ch for i, ch in enumerate(word) if i == 0 or ch != word[i - 1])

### Module: `idgham.py`
Mushaf-style rendering (idgham) so that corrections taken from the Quran look like the printed text.

In [ ]:
%%writefile idgham.py

from __future__ import annotations

import re

_MARKER = re.compile(r"^\(\d+\)$")

_SUKUN, _SHADDA, _FATHATAN = "\u0652", "\u0651", "\u064B"
_TANWEEN = set("\u064B\u064C\u064D")
_IDGHAM_AFTER_NOON = set("نمرل")
_IDGHAM_AFTER_LAM = set("لر")
_DIACRITIC_CHARS = set(
    "\u0610\u0611\u0612\u0613\u0614\u0615\u0616\u0617\u0618\u0619\u061A"
    "\u064B\u064C\u064D\u064E\u064F\u0650\u0651\u0652\u0670"
    "\u06D6\u06D7\u06D8\u06D9\u06DA\u06DB\u06DC\u06DF\u06E0\u06E1\u06E2\u06E3\u06E4"
    "\u06E7\u06E8\u06EA\u06EB\u06EC\u06ED"
)


def _base_letters(word: str) -> str:
    return "".join(c for c in word if c not in _DIACRITIC_CHARS)


def _insert_shadda(word: str) -> str:
    return word if not word else word[0] + _SHADDA + word[1:]


def _ends_with_tanween(word: str) -> bool:
    if not word:
        return False
    if word[-1] in _TANWEEN:
        return True
    return len(word) >= 2 and word[-1] in "اى" and word[-2] == _FATHATAN


def apply_idgham(text: str, extended: bool = True) -> str:
    words = text.split(" ")
    noon_set = _IDGHAM_AFTER_NOON if extended else set("نم")
    i = 0
    while i < len(words):
        word = words[i]
        if _MARKER.match(word) or not word:
            i += 1
            continue
        j = i + 1
        while j < len(words) and _MARKER.match(words[j]):
            j += 1
        if j < len(words):
            base = _base_letters(words[j])
            first = base[0] if base else ""
            if word.endswith("\u0646" + _SUKUN) and first in noon_set:
                words[i], words[j] = word[:-1], _insert_shadda(words[j])
            elif _ends_with_tanween(word) and first in noon_set:
                words[j] = _insert_shadda(words[j])
            elif word.endswith("\u0645" + _SUKUN) and first == "\u0645":
                words[i], words[j] = word[:-1], _insert_shadda(words[j])
            elif extended and word.endswith("\u0644" + _SUKUN) and first in _IDGHAM_AFTER_LAM:
                words[i], words[j] = word[:-1], _insert_shadda(words[j])
        i += 1
    return " ".join(words)

### Module: `alignment.py`
Word-level alignment between a quotation and a source: sliding window, longest common subsequence and a dynamic gap.

In [ ]:
%%writefile alignment.py

from __future__ import annotations

from difflib import SequenceMatcher
from math import ceil
from typing import Container, Dict, List, Optional, Sequence, Tuple

from idgham import apply_idgham
from normalization import aligned_words, vowel_signature


def lcs_length(a: Sequence[str], b: Sequence[str]) -> int:
    m, n = len(a), len(b)
    if m == 0 or n == 0:
        return 0
    if m < n:
        a, b, m, n = b, a, n, m
    prev = [0] * (n + 1)
    for i in range(m):
        curr = [0] * (n + 1)
        for j in range(n):
            curr[j + 1] = prev[j] + 1 if a[i] == b[j] else max(curr[j], prev[j + 1])
        prev = curr
    return prev[n]


def allowed_gap(n_tokens: int) -> int:
    return 0 if n_tokens <= 3 else max(1, ceil(0.2 * n_tokens))


def find_window(quote: List[str], source: List[str], gap: int) -> Tuple[int, int]:
    n, window = len(quote), len(quote) + gap
    if len(source) <= window + 2 * gap + 8:
        return 0, len(source)
    quote_set = set(quote)
    prefix = [0]
    for word in source:
        prefix.append(prefix[-1] + (word in quote_set))
    starts = sorted(range(len(source) - window + 1), key=lambda i: prefix[i + window] - prefix[i], reverse=True)[:8]
    best_start = max(starts, key=lambda i: lcs_length(quote, source[i : i + window]))
    return max(0, best_start - gap), min(len(source), best_start + window + gap)


def contains_sequence(source: Sequence[str], quote: Sequence[str]) -> bool:
    n = len(quote)
    return n > 0 and any(source[i : i + n] == list(quote) for i in range(len(source) - n + 1))


def locate_region(q_norm: List[str], s_norm: List[str], gap: int, slack: Optional[int] = None):
    slack = gap if slack is None else slack
    lo, hi = find_window(q_norm, s_norm, gap)
    matcher = SequenceMatcher(None, q_norm, s_norm[lo:hi], autojunk=False)
    blocks = [b for b in matcher.get_matching_blocks() if b.size]
    if blocks:
        first, last = blocks[0], blocks[-1]
        base = lo
        lo = base + max(0, first.b - first.a - slack)
        hi = min(hi, base + last.b + last.size + (len(q_norm) - last.a - last.size) + slack)
        matcher = SequenceMatcher(None, q_norm, s_norm[lo:hi], autojunk=False)
    return lo, hi, matcher


def best_region(quote: str, source: str) -> str:
    q_pairs, s_pairs = aligned_words(quote), aligned_words(source)
    if not q_pairs or len(s_pairs) <= len(q_pairs) + allowed_gap(len(q_pairs)) + 2:
        return source
    q_norm, s_norm = [p[1] for p in q_pairs], [p[1] for p in s_pairs]
    lo, hi, _ = locate_region(q_norm, s_norm, allowed_gap(len(q_norm)), slack=0)
    return " ".join(p[0] for p in s_pairs[lo:hi]) if hi > lo else source


def _diacritic_notes(quote_words: List[str], source_words: List[str], idgham_words: List[str]) -> List[dict]:
    notes = []
    for q_word, s_word, g_word in zip(quote_words, source_words, idgham_words):
        q_sig = vowel_signature(q_word)
        if not any(marks for _, marks in q_sig):
            continue
        conflict = True
        for variant in (s_word, g_word):
            v_sig = vowel_signature(variant)
            if len(v_sig) == len(q_sig) and all(set(qm) <= set(vm) for (_, qm), (_, vm) in zip(q_sig, v_sig)):
                conflict = False
                break
        if conflict:
            notes.append({"word": q_word, "source_word": s_word})
    return notes


def _orthographic_variant(q_tokens: Sequence[str], s_tokens: Sequence[str], vocabulary: Optional[Container[str]]) -> bool:
    if "".join(q_tokens) == "".join(s_tokens):
        return True
    if vocabulary is not None and len(q_tokens) == 1 and len(s_tokens) == 1:
        q, s = q_tokens[0], s_tokens[0]
        return q.replace("ا", "") == s.replace("ا", "") and q not in vocabulary
    return False


def align(quote: str, source: str, vocabulary: Optional[Container[str]] = None) -> Dict[str, object]:
    q_pairs, s_pairs = aligned_words(quote), aligned_words(source)
    idgham_pairs = aligned_words(apply_idgham(source)) if len(source) < 20000 else s_pairs
    if len(idgham_pairs) != len(s_pairs):
        idgham_pairs = s_pairs
    q_orig, q_norm = [p[0] for p in q_pairs], [p[1] for p in q_pairs]
    s_orig, s_norm = [p[0] for p in s_pairs], [p[1] for p in s_pairs]
    g_orig = [p[0] for p in idgham_pairs]

    gap = allowed_gap(len(q_norm))
    lo, hi, matcher = locate_region(q_norm, s_norm, gap)

    raw_ops = [(tag, i1, i2, j1 + lo, j2 + lo) for tag, i1, i2, j1, j2 in matcher.get_opcodes()]
    deleted = {w for tag, i1, i2, _, _ in raw_ops if tag in ("delete", "replace") for w in q_norm[i1:i2]}
    for position in (0, -1):
        if len(raw_ops) > 1 and raw_ops[position][0] == "insert" and not (set(s_norm[raw_ops[position][3]:raw_ops[position][4]]) & deleted):
            raw_ops.pop(position)

    ops, quote_side, source_side, missing, extra, notes = [], [], [], [], [], []
    mismatches = 0
    orthographic = 0
    for position, (tag, i1, i2, j1, j2) in enumerate(raw_ops):
        source_idx = list(range(j1, j2))
        if tag == "replace" and _orthographic_variant(q_norm[i1:i2], s_norm[j1:j2], vocabulary):
            tag, orthographic = "equal", orthographic + 1  
        if tag == "replace" and position in (0, len(raw_ops) - 1) and len(raw_ops) > 1 and (j2 - j1) > (i2 - i1):
            keep = [j for j in source_idx if s_norm[j] in set(q_norm[i1:i2]) | deleted]
            source_idx = keep
            tag = "replace" if keep else "delete"
        ops.append({"op": tag, "span": " ".join(q_orig[i1:i2]), "source": " ".join(s_orig[j] for j in source_idx)})
        if tag == "equal":
            notes += _diacritic_notes(q_orig[i1:i2], s_orig[j1:j2], g_orig[j1:j2])
        else:
            mismatches += max(i2 - i1, len(source_idx))
            quote_side += q_norm[i1:i2]
            source_side += [s_norm[j] for j in source_idx]
            extra += q_orig[i1:i2] if tag in ("delete", "replace") else []
            missing += [s_orig[j] for j in source_idx] if tag in ("insert", "replace") else []

    matched = sum(i2 - i1 for tag, i1, i2, _, _ in raw_ops if tag == "equal")
    exact = bool(q_norm) and mismatches == 0
    return {
        "word_similarity": round(2 * matched / max(len(q_norm) + (raw_ops[-1][4] - raw_ops[0][3] if raw_ops else 0), 1), 3),
        "word_diff": ops,
        "missing_from_span": missing,
        "extra_in_span": extra,
        "source_excerpt": " ".join(op["source"] for op in ops if op["source"]),
        "exact": exact,
        "mismatches": mismatches,
        "allowed_gap": gap,
        "near": (not exact) and matched > 0 and mismatches <= gap,
        "reordered": bool(quote_side) and sorted(quote_side) == sorted(source_side),
        "diacritic_notes": notes,
        "orthographic_variants": orthographic,
    }

### Module: `similarity.py`
Similarity indicators (coverage, token overlap, edit similarity ...) computed on the matched region only.

In [ ]:
%%writefile similarity.py
from __future__ import annotations

import re
import unicodedata
from typing import Dict, List

from alignment import best_region, lcs_length
from normalization import normalize_lenient, normalize_strict, tokenize

try:  
    from rapidfuzz.distance import Levenshtein as _RapidLevenshtein
except ImportError:  
    _RapidLevenshtein = None


def _edit_similarity(a: str, b: str, max_len: int = 600) -> float:
    a, b = a[:max_len], b[:max_len]
    if a == b:
        return 1.0
    if not a or not b:
        return 0.0
    if _RapidLevenshtein is not None:
        return float(_RapidLevenshtein.normalized_similarity(a, b))
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a):
        curr = [i + 1]
        for j, cb in enumerate(b):
            curr.append(min(curr[j] + 1, prev[j + 1] + 1, prev[j] + (ca != cb)))
        prev = curr
    return 1.0 - prev[-1] / max(len(a), len(b))


def _light_normalize(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[،؛؟!.,:;'\"()\[\]{}<>«»\-_/\\|@#$%^&*+=~`\u0640]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def compute_signals(claim: str, candidate: str, content_type: str = "Ayah", local: bool = True) -> Dict[str, float]:
    is_quran = content_type == "Ayah"
    full_candidate = candidate
    if local:   
        candidate = best_region(claim, candidate)
    normalize = normalize_strict if is_quran else normalize_lenient
    norm_claim, norm_cand = normalize(claim), normalize(candidate)

    claim_set, cand_set = set(tokenize(norm_claim)), set(tokenize(norm_cand))
    if claim_set and cand_set:
        shared = claim_set & cand_set
        token_overlap = len(shared) / len(claim_set | cand_set)
        coverage = len(shared) / len(claim_set)
    else:
        token_overlap = coverage = 0.0

    claim_tokens, cand_tokens = tokenize(norm_claim), tokenize(norm_cand)
    lcs_ratio = lcs_length(claim_tokens, cand_tokens) / len(claim_tokens) if claim_tokens else 0.0
    edit_sim = _edit_similarity(norm_claim, norm_cand)

    claim_chars, cand_chars = set(norm_claim.replace(" ", "")), set(norm_cand.replace(" ", ""))
    char_overlap = len(claim_chars & cand_chars) / len(claim_chars | cand_chars) if (claim_chars or cand_chars) else 0.0

    claim_flat, cand_flat = norm_claim.replace(" ", ""), norm_cand.replace(" ", "")
    is_substring = int(bool(claim_flat) and bool(cand_flat) and (claim_flat in cand_flat or cand_flat in claim_flat))

    diacritic_sim = (
        _edit_similarity(_light_normalize(claim), _light_normalize(candidate), max_len=800) if is_quran else edit_sim
    )
    short = len(claim_tokens) < 4

    if is_quran:
        w = (
            dict(coverage=0.35, diacritic_sim=0.30, lcs_ratio=0.15, token_overlap=0.10, char_overlap=0.05, edit_sim=0.05)
            if short
            else dict(coverage=0.25, diacritic_sim=0.30, lcs_ratio=0.20, token_overlap=0.10, char_overlap=0.05, edit_sim=0.10)
        )
        composite = (
            w["coverage"] * coverage + w["diacritic_sim"] * diacritic_sim + w["lcs_ratio"] * lcs_ratio
            + w["token_overlap"] * token_overlap + w["char_overlap"] * char_overlap + w["edit_sim"] * edit_sim
        )
        if is_substring and diacritic_sim >= 0.60:
            composite = max(composite, 0.88)
        elif is_substring:
            composite = max(composite, 0.75)
    else:
        w = (
            dict(coverage=0.40, lcs_ratio=0.20, token_overlap=0.20, char_overlap=0.10, edit_sim=0.10)
            if short
            else dict(coverage=0.30, lcs_ratio=0.28, token_overlap=0.18, char_overlap=0.12, edit_sim=0.12)
        )
        composite = (
            w["coverage"] * coverage + w["lcs_ratio"] * lcs_ratio + w["token_overlap"] * token_overlap
            + w["char_overlap"] * char_overlap + w["edit_sim"] * edit_sim
        )
        if is_substring:
            composite = max(composite, 0.82)

    return {
        "token_overlap": round(token_overlap, 4),
        "coverage": round(coverage, 4),
        "lcs_ratio": round(lcs_ratio, 4),
        "edit_sim": round(edit_sim, 4),
        "diacritic_sim": round(diacritic_sim, 4),
        "char_overlap": round(char_overlap, 4),
        "is_substring": is_substring,
        "source_fraction": round(min(1.0, len(claim_tokens) / max(len(tokenize(normalize(full_candidate))), 1)), 4),
        "composite": round(composite, 4),
    }


def best_match_score(claim: str, candidates: List[dict], content_type: str = "Ayah", local: bool = True):
    """Return ``(score, candidate_with_signals)`` for the best-scoring candidate."""
    best_score, best_candidate = 0.0, None
    for candidate in candidates:
        signals = compute_signals(claim, candidate.get("text", ""), content_type, local)
        if signals["composite"] > best_score:
            best_score, best_candidate = signals["composite"], {**candidate, "signals": signals}
    return best_score, best_candidate

### Module: `index_builder.py`
Search indexes: BM25 postings and phonetic n-gram anchors, built once from the corpora.

In [ ]:
%%writefile index_builder.py

from __future__ import annotations

import gzip
import json
import math
import pickle
import sys
import time
import zlib
from array import array
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

from normalization import content_words, normalize_for_matching, normalize_strict, phonetic_key, tokenize

INDEX_VERSION = 3
BM25_K1, BM25_B = 1.2, 0.75


class BM25Index:

    def __init__(self, postings: Dict[str, Tuple[array, array]], doc_len: array) -> None:
        self.postings, self.doc_len = postings, doc_len
        self.n_docs = len(doc_len)
        self.avg_len = (sum(doc_len) / self.n_docs) if self.n_docs else 1.0
        self.idf = {t: math.log(1 + (self.n_docs - len(d) + 0.5) / (len(d) + 0.5)) for t, (d, _) in postings.items()}

    def search(self, terms: Sequence[str], top_k: int) -> List[Tuple[int, float]]:
        scores: Dict[int, float] = defaultdict(float)
        for term in set(terms):
            entry = self.postings.get(term)
            if entry is None:
                continue
            idf = self.idf[term]
            for doc, tf in zip(*entry):
                norm = 1 - BM25_B + BM25_B * self.doc_len[doc] / self.avg_len
                scores[doc] += idf * tf * (BM25_K1 + 1) / (tf + BM25_K1 * norm)
        return sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_k]


def _postings(documents: List[List[str]]) -> Tuple[Dict[str, Tuple[array, array]], array]:
    docs: Dict[str, array] = defaultdict(lambda: array("I"))
    tfs: Dict[str, array] = defaultdict(lambda: array("H"))
    doc_len = array("I")
    for doc_id, tokens in enumerate(documents):
        doc_len.append(len(tokens))
        for term, tf in Counter(tokens).items():
            docs[term].append(doc_id)
            tfs[term].append(min(tf, 65535))
    return {term: (docs[term], tfs[term]) for term in docs}, doc_len


def _read_json(path: Path) -> list:
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as handle:
        data = json.load(handle)
    if not isinstance(data, list):
        raise ValueError(f"Unexpected corpus format in {path}: expected a JSON list")
    return data



def anchor_keys(tokens: Sequence[str]) -> List[Tuple[int, int]]:

    keys = [phonetic_key(t) for t in tokens]
    out = []
    for i in range(len(keys) - 2):
        out.append((zlib.crc32(f"{keys[i]} {keys[i + 1]} {keys[i + 2]}".encode()), i))
        if i + 3 < len(keys):
            out.append((zlib.crc32(f"{keys[i]} {keys[i + 1]} _ {keys[i + 3]}".encode()), i))
            out.append((zlib.crc32(f"{keys[i]} _ {keys[i + 2]} {keys[i + 3]}".encode()), i))
    return out


def build_quran_index(path: Path) -> dict:
    records, norm, word_count = [], [], []
    all_index: Dict[str, List[int]] = defaultdict(list)
    by_surah: Dict[int, Dict[int, int]] = defaultdict(dict)
    documents: List[List[str]] = []
    anchors: Dict[int, List[Tuple[int, int]]] = defaultdict(list)
    for entry in _read_json(Path(path)):
        text = (entry.get("ayah_text") or "").strip()
        if not text:
            continue
        idx = len(records)
        records.append({"surah_id": entry.get("surah_id"), "surah_name": entry.get("surah_name", ""),
                        "ayah_id": entry.get("ayah_id"), "text": text})
        by_surah[entry.get("surah_id")][entry.get("ayah_id")] = idx
        norm.append(normalize_for_matching(text))
        strict_tokens = tokenize(normalize_strict(text))
        word_count.append(len(strict_tokens))
        for word in set(strict_tokens):
            all_index[word].append(idx)
        documents.append(content_words(norm[-1].split()))
        for key, pos in anchor_keys(norm[-1].split()):
            anchors[key].append((idx, pos))
    postings, doc_len = _postings(documents)
    return {"version": INDEX_VERSION, "records": records, "norm": norm, "word_count": word_count,
            "all_index": dict(all_index), "by_surah": dict(by_surah), "postings": postings, "doc_len": doc_len,
            "anchors": dict(anchors)}


def build_hadith_index(path: Path) -> dict:
    records, documents = [], []
    for entry in _read_json(Path(path)):
        if not entry:
            continue
        matn = (entry.get("Matn") or "").strip() or None
        full = (entry.get("hadithTxt") or "").strip() or None
        if not (matn or full):
            continue

        records.append({"hadithID": entry.get("hadithID"), "book": entry.get("BookID"), "title": entry.get("title"),
                        "matn": matn, "full": None if matn else full})
        words = content_words(normalize_for_matching(full or matn).split())
        if matn and full:   
            extra = set(content_words(normalize_for_matching(matn).split())) - set(words)
            words += sorted(extra)
        documents.append(words)
    postings, doc_len = _postings(documents)
    return {"version": INDEX_VERSION, "records": records, "postings": postings, "doc_len": doc_len}


def save_index(index: dict, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(path, "wb", compresslevel=9) as handle:
        pickle.dump(index, handle, protocol=pickle.HIGHEST_PROTOCOL)


def load_index(path: Path) -> dict:
    with gzip.open(path, "rb") as handle:
        index = pickle.load(handle)   
    if index.get("version") != INDEX_VERSION:
        raise ValueError("Index version mismatch; rebuild with index_builder.py")
    return index


def main() -> None:
    base = Path(__file__).resolve().parent
    out = base / "index"
    for name, builder, source in (("quran", build_quran_index, base / "data" / "quran.json"),
                                  ("hadith", build_hadith_index, base / "data" / "hadith.json")):
        started = time.time()
        source = source if source.is_file() else source.with_name(source.name + ".gz")
        index = builder(source)
        save_index(index, out / f"{name}.idx.gz")
        size = (out / f"{name}.idx.gz").stat().st_size / 1e6
        print(f"{name}: {len(index['records'])} records -> index/{name}.idx.gz ({size:.1f} MB, {time.time() - started:.1f}s)")


if __name__ == "__main__":
    sys.exit(main())

### Module: `retrieval.py`
Retriever: BM25 recall and character n-gram re-ranking over the Quran and Hadith indexes.

In [ ]:
%%writefile retrieval.py

from __future__ import annotations

import logging
import threading
from collections import OrderedDict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

from index_builder import BM25Index, build_hadith_index, build_quran_index, load_index, save_index
from normalization import char_ngrams, content_words, normalize_for_matching, normalize_lenient, normalize_strict, tokenize

logger = logging.getLogger(__name__)

BASE_DIR = Path(__file__).resolve().parent
DATA_DIR = BASE_DIR / "data"
INDEX_DIR = BASE_DIR / "index"


class CorpusError(RuntimeError):
    """Raised when a corpus or index file is missing or malformed."""


class SourceRetriever:
    

    def __init__(self, index_dir: Path = INDEX_DIR, data_dir: Path = DATA_DIR) -> None:
        self.index_dir, self.data_dir = Path(index_dir), Path(data_dir)
        self._hadith_lock = threading.Lock()
        self._hadith: Optional[dict] = None
        self._norm_cache: "OrderedDict[Tuple[int, str], str]" = OrderedDict()

        quran = self._load("quran", build_quran_index)
        self.quran: List[dict] = quran["records"]
        self.q_norm_match: List[str] = quran["norm"]
        self.q_word_count: List[int] = quran["word_count"]
        self.q_all_index: Dict[str, List[int]] = quran["all_index"]
        self.quran_by_surah: Dict[int, Dict[int, int]] = quran["by_surah"]
        self.quran_bm25 = BM25Index(quran["postings"], quran["doc_len"])
        self.quran_anchors: Dict[int, List[Tuple[int, int]]] = quran["anchors"]
        logger.info("Quran index ready: %d ayahs", len(self.quran))

    
    def _load(self, name: str, builder) -> dict:
        path = self.index_dir / f"{name}.idx.gz"
        if path.is_file():
            try:
                return load_index(path)
            except Exception:
                logger.warning("Index %s is unreadable; rebuilding from data/", path)
        source = self.data_dir / ("quran.json" if name == "quran" else "hadith.json")
        gz = source.with_name(source.name + ".gz")
        source = source if source.is_file() else gz
        if not source.is_file():
            raise CorpusError(f"Neither {path} nor the source corpus {source} was found")
        index = builder(source)
        try:
            save_index(index, path)
        except OSError:
            logger.info("Could not cache %s (read-only file system); continuing in memory", path)
        return index

    def warm(self) -> None:
        _ = self.hadith_data

    @property
    def hadith_loaded(self) -> bool:
        return self._hadith is not None

    @property
    def hadith_data(self) -> dict:
        if self._hadith is None:
            with self._hadith_lock:
                if self._hadith is None:
                    data = self._load("hadith", build_hadith_index)
                    data["bm25"] = BM25Index(data["postings"], data["doc_len"])
                    self._hadith = data
                    logger.info("Hadith index ready: %d records", len(data["records"]))
        return self._hadith

    @property
    def hadith(self) -> List[dict]:
        return self.hadith_data["records"]

    @property
    def quran_vocabulary(self) -> frozenset:
        if getattr(self, "_vocab", None) is None:
            self._vocab = frozenset(word for text in self.q_norm_match for word in text.split())
        return self._vocab

    def search_quran_ayahs(self, query: str, top_k: int = 25, extra_bm25: int = 10) -> List[dict]:
        query_words = tokenize(normalize_strict(query))
        if not query_words:
            return []
        votes: Dict[int, int] = {}
        for word in query_words:
            for idx in self.q_all_index.get(word, ()):
                votes[idx] = votes.get(idx, 0) + 1
        scored: List[Tuple[int, float]] = []
        for idx, vote in votes.items():
            coverage = vote / len(query_words)
            precision = vote / self.q_word_count[idx] if self.q_word_count[idx] else 0.0
            f1 = 2 * coverage * precision / (coverage + precision) if coverage + precision > 0 else 0.0
            scored.append((idx, f1))
        scored.sort(key=lambda item: item[1], reverse=True)
        scores = dict(scored)
        ranked = [idx for idx, _ in scored[:top_k]]
        seen = set(ranked)
        bm25_hits = self.quran_bm25.search(content_words(normalize_for_matching(query).split()), extra_bm25)
        ranked += [idx for idx, _ in bm25_hits if idx not in seen]
        results = []
        for idx in ranked:
            candidate = dict(self.quran[idx])
            candidate.update(type="Quran", retrieval_score=scores.get(idx, 0.0))
            results.append(candidate)
        return results

    def quran_seed_ayahs(self, query_words: Sequence[str], top_k: int = 25) -> List[int]:
        return [idx for idx, _ in self.quran_bm25.search(query_words, top_k)]

    def hadith_candidates(self, query_words: Sequence[str], top_k: int) -> List[int]:
        return [idx for idx, _ in self.hadith_data["bm25"].search(query_words, top_k)]

    def hadith_norm(self, idx: int, field: str) -> Optional[str]:
        record = self.hadith[idx]
        raw = record.get(field)
        if not raw:
            return None
        key = (idx, field)
        if key in self._norm_cache:
            self._norm_cache.move_to_end(key)
            return self._norm_cache[key]
        value = normalize_for_matching(raw)
        self._norm_cache[key] = value
        if len(self._norm_cache) > 4096:
            self._norm_cache.popitem(last=False)
        return value

    def search_hadith(self, query: str, top_k: int = 15, pool: int = 60) -> List[dict]:
        words = content_words(normalize_for_matching(query).split())
        if not words:
            return []
        query_grams = char_ngrams(normalize_lenient(query))
        results = []
        for idx in self.hadith_candidates(words, pool):
            record = self.hadith[idx]
            text = record["matn"] or record["full"]
            doc_grams = char_ngrams(normalize_lenient(text))
            cosine = (
                len(query_grams & doc_grams) / ((len(query_grams) * len(doc_grams)) ** 0.5)
                if query_grams and doc_grams
                else 0.0
            )
            results.append(
                {
                    "type": "Hadith",
                    "idx": idx,
                    "hadithID": record["hadithID"],
                    "book": record["book"],
                    "title": record["title"],
                    "text": text,
                    "has_matn": bool(record["matn"]),
                    "retrieval_score": cosine,
                }
            )
        results.sort(key=lambda c: c["retrieval_score"], reverse=True)
        return results[:top_k]

### Module: `detector.py`
Rule-based detector: quotation marks and brackets, typed by the corpora (introductory phrases are only hints).

In [ ]:
%%writefile detector.py

from __future__ import annotations

import re
from dataclasses import dataclass
from typing import List, Optional

from normalization import normalize_for_matching, normalize_lenient, normalize_strict, tokenize

@dataclass
class DetectedSpan:
    start: int
    end: int                        
    label: str                      
    confidence: Optional[float]     
    source: str                     
    text: str = ""
    hint: Optional[str] = None      


QUOTE_CHARS = " \t\r\n\"“”«»﴿﴾{}()[]"


def trim_span(text: str, start: int, end: int):
    while start < end and text[start] in QUOTE_CHARS:
        start += 1
    while end > start and text[end - 1] in QUOTE_CHARS + ".،,؛:":
        end -= 1
    return start, end


AYAH_TRIGGERS = [
    "قال الله", "قوله تعالى", "قال تعالى", "يقول الله", "يقول تعالى", "قال سبحانه", "قوله سبحانه", "في كتابه",
    "سورة", "الآية", "الاية", "الآيات", "القرآن", "القران", "كتاب الله", "عز وجل", "جل جلاله", "فقال تعالى",
    "ذكر الله", "﴿",
]
HADITH_TRIGGERS = [
    "رسول الله", "النبي", "صلى الله عليه وسلم", "ﷺ", "عليه الصلاة والسلام", "حديث", "رواه", "روى", "متفق عليه",
    "الحديث", "فقال", "قال ص", "صلى الله عليه", "وسلم",
]
_FORMULA_WORDS = {
    normalize_for_matching(w)
    for w in "قال قالت رسول الله صلى عليه وسلم النبي تعالى سبحانه عز وجل فقال يقول الكريم الشريف الحديث الآية روى رواه عن أن أنه البخاري ومسلم".split()
}
_BRACKET_PAIRS = [("“", "”"), ("«", "»"), ("﴿", "﴾"), ("{", "}"), ("(", ")"), ("[", "]")]


class RuleDetector:

    def __init__(self, retriever=None, min_words: int = 3, context_chars: int = 110,
                 min_corpus_cov: float = 0.6, decouple_triggers: bool = True, quoted_min_cov: float = 0.5,
                 quoted_min_cov_hadith: float = 0.75) -> None:
        self.kb, self.min_words, self.context_chars, self.min_corpus_cov = retriever, min_words, context_chars, min_corpus_cov
        self.decouple_triggers = decouple_triggers and retriever is not None
        self.quoted_min_cov, self.quoted_min_cov_hadith = quoted_min_cov, quoted_min_cov_hadith

    @staticmethod
    def _segments(text: str):
        segments = []
        quote_positions = [m.start() for m in re.finditer('"', text)]
        if len(quote_positions) % 2 == 0:
            pairs = zip(quote_positions[0::2], quote_positions[1::2])   
        else:   
            pairs = zip(quote_positions, quote_positions[1:])
        for a, b in pairs:
            segments.append((a + 1, b))
        for opener, closer in _BRACKET_PAIRS:
            for m in re.finditer(re.escape(opener) + r"(.*?)" + re.escape(closer), text, re.S):
                segments.append((m.start(1), m.end(1)))
        return segments

    @staticmethod
    def _trigger_type(context: str) -> Optional[str]:
        best_end, best_label = -1, None
        for label, triggers in (("Ayah", AYAH_TRIGGERS), ("Hadith", HADITH_TRIGGERS)):
            for trigger in triggers:
                pos = context.rfind(trigger)
                if pos >= 0 and pos + len(trigger) > best_end:
                    best_end, best_label = pos + len(trigger), label
        return best_label

    def _corpus_coverage(self, span: str):
        if self.kb is None:
            return 0.0, 0.0
        quran_words = set(tokenize(normalize_strict(span)))
        if not quran_words:
            return 0.0, 0.0
        quran_cov = self._quran_window_coverage(quran_words, span)
        hadith_words = set(tokenize(normalize_lenient(span)))
        hadith_cov = max(
            (len(hadith_words & set(tokenize(normalize_lenient(c["text"])))) / len(hadith_words)
             for c in self.kb.search_hadith(span, top_k=5)),
            default=0.0,
        ) if hadith_words else 0.0
        return quran_cov, hadith_cov

    def _quran_window_coverage(self, quran_words: set, span: str) -> float:
        kb, best = self.kb, 0.0
        for cand in kb.search_quran_ayahs(span, top_k=5):
            surah = kb.quran_by_surah.get(cand["surah_id"], {})
            for first in range(cand["ayah_id"] - 2, cand["ayah_id"] + 1):
                words: set = set()
                for length in (1, 2, 3):
                    idx = surah.get(first + length - 1)
                    if idx is None:
                        break
                    ayah_words = set(tokenize(normalize_strict(kb.quran[idx]["text"])))
                    if length > 1 and not any(len(w) >= 4 for w in quran_words & ayah_words):
                        break   
                    words |= ayah_words
                    if first <= cand["ayah_id"] <= first + length - 1:
                        best = max(best, len(quran_words & words) / len(quran_words))
        return best

    def _label_from_corpus(self, n_words: int, trigger: Optional[str], quran_cov: float, hadith_cov: float) -> Optional[str]:
        quran_ok = quran_cov >= self.quoted_min_cov
        hadith_ok = hadith_cov >= self.quoted_min_cov_hadith   
        if (quran_ok or hadith_ok) and n_words >= 4:
            if trigger and ((quran_ok if trigger == "Ayah" else hadith_ok)) and abs(quran_cov - hadith_cov) < 0.25:
                return trigger
            if quran_ok and hadith_ok:
                return "Ayah" if quran_cov >= hadith_cov else "Hadith"
            return "Ayah" if quran_ok else "Hadith"
        return trigger   

    def detect(self, text: str) -> List[DetectedSpan]:
        candidates = []
        for start, end in self._segments(text):
            start, end = trim_span(text, start, end)
            if end <= start:
                continue
            inner = text[start:end]
            words = [w for w in normalize_for_matching(inner).split() if w]
            if len(words) < 2 or len(inner) > 3000:
                continue
            trigger = self._trigger_type(text[max(0, start - self.context_chars):start])
            if len(words) < self.min_words and not trigger:  
                continue
            if sum(w in _FORMULA_WORDS for w in words) / len(words) >= 0.6:
                continue
            quran_cov, hadith_cov = self._corpus_coverage(inner)
            label = None
            if self.decouple_triggers:
                label = self._label_from_corpus(len(words), trigger, quran_cov, hadith_cov)
            elif trigger:
                label = trigger
                other, mine = (hadith_cov, quran_cov) if trigger == "Ayah" else (quran_cov, hadith_cov)
                if other >= 0.8 and mine < 0.5:
                    label = "Hadith" if trigger == "Ayah" else "Ayah"
            elif max(quran_cov, hadith_cov) >= self.min_corpus_cov and len(words) >= 4:
                label = "Ayah" if quran_cov >= hadith_cov else "Hadith"
            if label is None:
                continue
            candidates.append((bool(trigger), max(quran_cov, hadith_cov), end - start, start, end, label, trigger))

        candidates.sort(key=lambda c: (c[0], c[1], c[2]), reverse=True)   
        taken = []
        for _, _, _, start, end, label, hint in candidates:
            if all(end <= t_start or start >= t_end for t_start, t_end, _, _ in taken):
                taken.append((start, end, label, hint))
        taken.sort()
        return [DetectedSpan(s, e, label, None, "rules", text[s:e], hint) for s, e, label, hint in taken]

### Module: `scanner.py`
Corpus scanner for unannounced quotations, whole-ayah matching and the hybrid detector.

In [ ]:
%%writefile scanner.py

from __future__ import annotations

import re
from collections import defaultdict
from dataclasses import dataclass
from difflib import SequenceMatcher
from typing import Dict, List, Optional, Sequence, Tuple

from alignment import aligned_words
from detector import DetectedSpan, RuleDetector, trim_span
from index_builder import anchor_keys
from normalization import content_words, normalize_for_matching, phonetic_key
from retrieval import SourceRetriever

_ARABIC = re.compile(r"[\u0621-\u064A]")
_STRONG_BOUNDARY = re.compile(r"[.؟?!؛;:\n…]")


@dataclass
class Token:
    skeleton: str
    key: str
    start: int
    end: int
    boundary_after: bool   


def tokenize_with_offsets(text: str) -> List[Token]:
    matches = list(re.finditer(r"\S+", text))
    tokens: List[Token] = []
    for i, match in enumerate(matches):
        skeleton = normalize_for_matching(match.group()).replace(" ", "")
        if not skeleton or not _ARABIC.search(skeleton):
            if tokens and _STRONG_BOUNDARY.search(match.group()):
                tokens[-1].boundary_after = True
            continue
        gap_end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        trailing = text[match.end():gap_end] + match.group()[-2:]
        tokens.append(Token(skeleton, phonetic_key(skeleton), match.start(), match.end(),
                            bool(_STRONG_BOUNDARY.search(trailing))))
    return tokens


@dataclass
class _Region:
    start: int          
    end: int            
    label: str
    matched: int
    ratio: float
    idf: float
    surah: Optional[int] = None
    first_ayah: Optional[int] = None
    last_ayah: Optional[int] = None


class CorpusScanner:
   

    def __init__(self, retriever: SourceRetriever, min_tokens: int = 4, min_ratio: float = 0.7, min_idf: float = 8.0,
                 hadith_min_tokens: int = 6, hadith_min_idf: float = 14.0, scan_hadith: bool = True) -> None:
        self.kb = retriever
        self.min_tokens, self.min_ratio, self.min_idf = min_tokens, min_ratio, min_idf
        self.hadith_min_tokens, self.hadith_min_idf = hadith_min_tokens, hadith_min_idf
        self.scan_hadith = scan_hadith

    def scan(self, text: str, exclude: Sequence[Tuple[int, int]] = ()) -> List[DetectedSpan]:
        tokens = tokenize_with_offsets(text)
        runs = self._runs(tokens, exclude)
        spans: List[DetectedSpan] = []
        for run in runs:
            for region in self._merge(self._scan_quran(run)):
                spans.append(self._to_span(text, run, region))
        if self.scan_hadith:
            for run in runs:
                taken = [(s.start, s.end) for s in spans]
                for region in self._scan_hadith(run, taken):
                    spans.append(self._to_span(text, run, region))
        return sorted(spans, key=lambda s: s.start)

    @staticmethod
    def _runs(tokens: List[Token], exclude: Sequence[Tuple[int, int]]) -> List[List[Token]]:
        runs, current = [], []
        for token in tokens:
            if any(token.start < e and token.end > s for s, e in exclude):
                if current:
                    runs.append(current)
                current = []
            else:
                current.append(token)
        if current:
            runs.append(current)
        return runs

    @staticmethod
    def _to_span(text: str, run: List[Token], region: _Region) -> DetectedSpan:
        start, end = trim_span(text, run[region.start].start, run[region.end - 1].end)
        return DetectedSpan(start, end, region.label, round(region.ratio, 3), "scan", text[start:end])

    
    def _scan_quran(self, run: List[Token]) -> List[_Region]:
        kb = self.kb
        if len(run) < self.min_tokens:
            return []
        hits: Dict[int, List[Tuple[int, int]]] = defaultdict(list)
        for key_hash, pos in anchor_keys([t.skeleton for t in run]):
            postings = kb.quran_anchors.get(key_hash)
            if not postings or len(postings) > 60:   
                continue
            for ayah, source_pos in postings:
                hits[ayah].append((pos, source_pos))

        regions: List[_Region] = self._whole_ayahs(run)
        for ayah, found in hits.items():
            found.sort()
            chains: List[dict] = []
            for pos, source_pos in found:
                diagonal = source_pos - pos
                for chain in chains:
                    if abs(diagonal - chain["d"]) <= 2 and pos - chain["last"] <= 5:
                        chain["hits"].append((pos, source_pos))
                        chain["last"], chain["d"] = pos, diagonal
                        break
                else:
                    chains.append({"d": diagonal, "last": pos, "hits": [(pos, source_pos)]})
            for chain in chains:
                region = self._grow(run, ayah, chain)
                if region is not None:
                    regions.append(region)
        return self._non_overlapping(regions)

    def _whole_ayah_index(self) -> Dict[str, list]:
        if getattr(self, "_whole", None) is None:
            index: Dict[str, list] = defaultdict(list)
            for i, norm in enumerate(self.kb.q_norm_match):
                words = tuple(w for w in norm.split() if w)
                if len(words) >= self.min_tokens:
                    index[words[0]].append((words, i))
            for entries in index.values():
                entries.sort(key=lambda e: -len(e[0]))   
            self._whole = index
        return self._whole

    def _whole_ayahs(self, run: List[Token]) -> List[_Region]:
        index, skeletons, found, i = self._whole_ayah_index(), [t.skeleton for t in run], [], 0
        while i < len(run):
            for words, ayah in index.get(skeletons[i], ()):
                n = len(words)
                if tuple(skeletons[i:i + n]) == words and not any(t.boundary_after for t in run[i:i + n - 1]):
                    record = self.kb.quran[ayah]
                    found.append(_Region(i, i + n, "Ayah", n, 1.0, 99.0, record["surah_id"], record["ayah_id"], record["ayah_id"]))
                    i += n - 1
                    break
            i += 1
        return found

    def _grow(self, run: List[Token], ayah: int, chain: dict) -> Optional[_Region]:
        kb = self.kb
        source = kb.q_norm_match[ayah].split()
        source_keys = [phonetic_key(w) for w in source]
        diagonal = sorted(source_pos - pos for pos, source_pos in chain["hits"])[len(chain["hits"]) // 2]
        start = min(pos for pos, _ in chain["hits"])
        end = min(len(run), max(pos for pos, _ in chain["hits"]) + 3)

        def key_at(i: int) -> Optional[str]:
            return source_keys[i + diagonal] if 0 <= i + diagonal < len(source_keys) else None

        while start > 0 and not run[start - 1].boundary_after:   
            if run[start - 1].key == key_at(start - 1):
                start -= 1
            elif start >= 2 and not run[start - 2].boundary_after and run[start - 2].key == key_at(start - 2):
                start -= 2   
            else:
                break
        while end < len(run) and not run[end - 1].boundary_after:
            if run[end].key == key_at(end):
                end += 1
            elif end + 1 < len(run) and run[end + 1].key == key_at(end + 1):
                end += 2
            else:
                break

        start = self._soft_left(run, start, diagonal, source_keys)
        end = self._soft_right(run, end, diagonal, source_keys)
        positions = range(start, end)
        matched_idx = [i for i in positions if run[i].key == key_at(i)]
        matched = len(matched_idx)
        n = end - start
        idf = sum(kb.quran_bm25.idf.get(run[i].skeleton, 0.0) for i in matched_idx)
        ratio = matched / n if n else 0.0
        needed_idf = self.min_idf if n >= 5 else self.min_idf + 6.0   
        if n < self.min_tokens or matched < self.min_tokens or ratio < self.min_ratio or idf < needed_idf:
            return None
        record = kb.quran[ayah]
        return _Region(start, end, "Ayah", matched, ratio, idf, record["surah_id"], record["ayah_id"], record["ayah_id"])

    @staticmethod
    def _closest(run: List[Token], candidates, target: str):

        scored = [(SequenceMatcher(None, "".join(t.key for t in run[a:b]), target).ratio(), a, b) for a, b in candidates]
        if not scored:
            return None
        top = max(score for score, _, _ in scored)
        if top < 0.6:
            return None
        return max((c for c in scored if c[0] >= top - 0.2), key=lambda c: c[2] - c[1])

    def _soft_left(self, run: List[Token], start: int, diagonal: int, source_keys: List[str]) -> int:
        missing = min(start + diagonal, 2)
        if missing <= 0:
            return start
        target = "".join(source_keys[start + diagonal - missing : start + diagonal])
        candidates = [(start - n, start) for n in range(max(1, missing - 1), missing + 2)
                      if start - n >= 0 and not any(run[i].boundary_after for i in range(start - n, start))]
        best = self._closest(run, candidates, target)
        return best[1] if best else start

    def _soft_right(self, run: List[Token], end: int, diagonal: int, source_keys: List[str]) -> int:
        missing = min(len(source_keys) - (end + diagonal), 2)
        if missing <= 0 or end >= len(run) or run[end - 1].boundary_after:
            return end
        target = "".join(source_keys[end + diagonal : end + diagonal + missing])
        candidates = [(end, end + n) for n in range(max(1, missing - 1), missing + 2)
                      if end + n <= len(run) and not any(run[i].boundary_after for i in range(end, end + n - 1))]
        best = self._closest(run, candidates, target)
        return best[2] if best else end

    @staticmethod
    def _non_overlapping(regions: List[_Region]) -> List[_Region]:
        chosen: List[_Region] = []
        for region in sorted(regions, key=lambda r: (r.matched, r.ratio), reverse=True):
            if all(region.end <= c.start or region.start >= c.end for c in chosen):
                chosen.append(region)
        return sorted(chosen, key=lambda r: r.start)

    @staticmethod
    def _merge(regions: List[_Region]) -> List[_Region]:
        merged: List[_Region] = []
        for region in regions:
            last = merged[-1] if merged else None
            if (last and last.surah == region.surah and region.first_ayah == last.last_ayah + 1
                    and region.start - last.end <= 1):
                last.end, last.matched = region.end, last.matched + region.matched
                last.ratio = last.matched / (last.end - last.start)
                last.idf += region.idf
                last.last_ayah = region.last_ayah
            else:
                merged.append(region)
        return merged

    def _scan_hadith(self, run: List[Token], taken: Sequence[Tuple[int, int]]) -> List[_Region]:
        kb = self.kb
        regions: List[_Region] = []
        segment_start = 0
        for i, token in enumerate(run):
            if token.boundary_after or i == len(run) - 1:
                segment = (segment_start, i + 1)
                segment_start = i + 1
                if segment[1] - segment[0] < self.hadith_min_tokens:
                    continue
                for a, b in self._windows(*segment):
                    region = self._match_hadith(run, a, b)
                    if region is not None:
                        regions.append(region)
        return self._non_overlapping(regions)

    @staticmethod
    def _windows(start: int, end: int, size: int = 24, stride: int = 12):
        if end - start <= 40:
            yield start, end
        else:
            for a in range(start, end - 6, stride):
                yield a, min(end, a + size)

    def _match_hadith(self, run: List[Token], a: int, b: int) -> Optional[_Region]:
        kb = self.kb
        window = run[a:b]
        words = content_words(t.skeleton for t in window)
        if len(words) < 4:
            return None
        window_keys = [t.key for t in window]
        best = None
        for idx in kb.hadith_candidates(words, 3):
            record = kb.hadith[idx]
            source = [p[1] for p in aligned_words(record["matn"] or record["full"])]
            matcher = SequenceMatcher(None, window_keys, [phonetic_key(w) for w in source], autojunk=False)
            blocks = [blk for blk in matcher.get_matching_blocks() if blk.size >= 3]
            matched = sum(blk.size for blk in blocks)
            if matched >= self.hadith_min_tokens and (best is None or matched > best[0]):
                best = (matched, blocks)
        if best is None:
            return None
        matched, blocks = best
        start, end = blocks[0].a, blocks[-1].a + blocks[-1].size
        ratio = matched / (end - start)
        idf = sum(kb.hadith_data["bm25"].idf.get(window[i].skeleton, 0.0) for blk in blocks for i in range(blk.a, blk.a + blk.size))
        if ratio < self.min_ratio or idf < self.hadith_min_idf:
            return None
        return _Region(a + start, a + end, "Hadith", matched, ratio, idf)


class HybridDetector:


    def __init__(self, retriever: SourceRetriever, use_scanner: bool = True, rules_use_corpus: bool = True,
                 decouple_triggers: bool = True) -> None:

        self.rules = RuleDetector(retriever if rules_use_corpus else None, decouple_triggers=decouple_triggers)
        self.scanner = CorpusScanner(retriever) if use_scanner else None

    def detect(self, text: str) -> List[DetectedSpan]:
        spans = self.rules.detect(text)
        if self.scanner is not None:
            found = sorted(self.scanner.scan(text, [(s.start, s.end) for s in spans]), key=lambda s: s.end - s.start, reverse=True)
            kept: List[DetectedSpan] = []
            for span in found:   
                if all(span.end <= k.start or span.start >= k.end for k in kept):
                    kept.append(span)
            spans += kept
        return sorted(spans, key=lambda s: s.start)

### Module: `verifier.py`
The decision engine: verification, source-backed correction, abstention and the anti-hallucination guard.

In [ ]:
%%writefile verifier.py

from __future__ import annotations

import difflib
import logging
import time
from dataclasses import dataclass, field
from typing import Dict, List, Optional

from alignment import align
from detector import DetectedSpan
from scanner import HybridDetector
from idgham import apply_idgham
from normalization import content_words, normalize_for_matching
from retrieval import SourceRetriever
from similarity import best_match_score, compute_signals

logger = logging.getLogger(__name__)

MAX_INPUT_CHARS = 20_000
SHORT_QUOTE_WORDS = 3   
MIN_EXACT_TOKENS = 3   



@dataclass
class VerifierConfig:
    

    quran_correct_threshold: float = 0.94
    quran_uncertain_low: float = 0.45
    quran_min_coverage: float = 0.40
    hadith_correct_threshold: float = 0.88
    hadith_uncertain_low: float = 0.30
    hadith_min_coverage: float = 0.70
    quran_top_k: int = 25
    hadith_top_k: int = 15
    hadith_retrieval_guard: float = 0.20


@dataclass
class CorrectorConfig:


    max_window: int = 8
    hadith_top_k: int = 40
    quran_strong: float = 0.65
    min_full_ratio: float = 0.40
    quran_low: float = 0.55
    hadith_strong: float = 1.01
    hadith_low: float = 0.45


@dataclass
class PipelineConfig:
    verifier: VerifierConfig = field(default_factory=VerifierConfig)
    corrector: CorrectorConfig = field(default_factory=CorrectorConfig)
    verified_min_conf: float = 0.75        
    unsupported_min_conf: float = 0.70     
    unsupported_strength: float = 0.35     



@dataclass
class Verification:
    verdict: str                         
    confidence: float
    best_score: float
    method: str
    source: Optional[dict] = None        
    n_candidates: int = 0
    retrieval_top: float = 0.0


class Verifier:
    

    def __init__(self, retriever: SourceRetriever, config: Optional[VerifierConfig] = None) -> None:
        self.kb = retriever
        self.cfg = config or VerifierConfig()

    def verify(self, span_text: str, content_type: str) -> Verification:
        if not span_text or not span_text.strip():
            return self._result("Incorrect", 0.95, 0.0, None, 0, "empty_span")
        if content_type == "Ayah":
            return self._verify_quran(span_text)
        if content_type == "Hadith":
            return self._verify_hadith(span_text)
        return self._result("Incorrect", 0.5, 0.0, None, 0, "unknown_type")

    def _verify_quran(self, span: str) -> Verification:
        cfg = self.cfg
        candidates = self.kb.search_quran_ayahs(span, top_k=cfg.quran_top_k)
        if not candidates:
            return self._result("Incorrect", 0.8, 0.0, None, 0, "no_candidates")
        score, best = best_match_score(span, candidates, "Ayah")
        signals = best.get("signals", {}) if best else {}
        coverage, is_substring = signals.get("coverage", 0.0), signals.get("is_substring", 0)
        n = len(candidates)

        if is_substring and coverage >= cfg.quran_min_coverage:
            return self._result("Correct", min(0.98, 0.85 + score * 0.15), score, best, n, "substring_match")
        if score >= cfg.quran_correct_threshold and coverage >= cfg.quran_min_coverage:
            return self._result("Correct", min(0.95, 0.70 + score * 0.25), score, best, n, "threshold_pass")
        if score <= cfg.quran_uncertain_low:
            return self._result("Incorrect", min(0.95, 0.70 + (1 - score) * 0.25), score, best, n, "threshold_fail")

        strong = sum(
            1 for cand in candidates[:10]
            if (s := compute_signals(span, cand.get("text", ""), "Ayah"))["coverage"] >= 0.80 and s["lcs_ratio"] >= 0.75
        )
        if strong >= 2:
            return self._result("Correct", 0.60 + min(0.20, strong * 0.05), score, best, n, "borderline_multi_cov")
        return self._result("Incorrect", 0.58, score, best, n, "borderline_default")

    def _verify_hadith(self, span: str) -> Verification:
        cfg = self.cfg
        candidates = self.kb.search_hadith(span, top_k=cfg.hadith_top_k)
        if not candidates:
            return self._result("Incorrect", 0.75, 0.0, None, 0, "no_candidates")
        top_retrieval = candidates[0].get("retrieval_score", 0.0)
        score, best = best_match_score(span, candidates, "Hadith")
        signals = best.get("signals", {}) if best else {}
        coverage, is_substring = signals.get("coverage", 0.0), signals.get("is_substring", 0)
        n = len(candidates)

        if is_substring and coverage >= cfg.hadith_min_coverage and top_retrieval >= cfg.hadith_retrieval_guard:
            return self._result("Correct", min(0.97, 0.80 + score * 0.17), score, best, n, "substring_match", top_retrieval)
        if score >= cfg.hadith_correct_threshold and coverage >= cfg.hadith_min_coverage:
            return self._result("Correct", min(0.92, 0.65 + score * 0.27), score, best, n, "threshold_pass", top_retrieval)
        if score <= cfg.hadith_uncertain_low:
            return self._result("Incorrect", min(0.90, 0.65 + (1 - score) * 0.25), score, best, n, "threshold_fail", top_retrieval)

        moderate = sum(
            1 for cand in candidates[:8]
            if (s := compute_signals(span, cand.get("text", ""), "Hadith"))["coverage"] >= 0.65 and s["lcs_ratio"] >= 0.55
        )
        if moderate >= 2 and top_retrieval >= 0.30:
            return self._result("Correct", 0.58 + min(0.22, moderate * 0.06), score, best, n, "borderline_multi_cov", top_retrieval)
        if top_retrieval < 0.25 or score < 0.45:
            return self._result("Incorrect", 0.60, score, best, n, "borderline_low_retrieval", top_retrieval)
        return self._result("Incorrect", 0.55, score, best, n, "borderline_default", top_retrieval)

    @staticmethod
    def _result(verdict, confidence, score, best, n_candidates, method, top_retrieval=0.0) -> Verification:
        return Verification(verdict, round(confidence, 4), round(score, 4), method, best, n_candidates, round(top_retrieval, 4))


@dataclass
class CorrectionMatch:
    kind: str                 
    strength: float            
    full_ratio: float
    text: str                  
    source: dict               
    display: str = ""          


class Corrector:
    def __init__(self, retriever: SourceRetriever, config: Optional[CorrectorConfig] = None) -> None:
        self.kb = retriever
        self.cfg = config or CorrectorConfig()

    def match(self, span_text: str, content_type: str) -> Optional[CorrectionMatch]:
        return self.match_quran(span_text) if content_type == "Ayah" else self.match_hadith(span_text)

    def match_quran(self, query_text: str) -> Optional[CorrectionMatch]:
        kb = self.kb
        query_norm = normalize_for_matching(query_text)
        query_words = content_words(query_norm.split())
        if not query_words:
            return None
        query_len = len(query_norm)
        memo: Dict[tuple, tuple] = {}
        best = None   
        for seed in kb.quran_seed_ayahs(query_words, top_k=25):
            surah, ayah = kb.quran[seed]["surah_id"], kb.quran[seed]["ayah_id"]
            ayahs = kb.quran_by_surah[surah]
            min_ayah, max_ayah = min(ayahs), max(ayahs)
            for offset in range(3):
                start = ayah - offset
                if start < min_ayah:
                    continue
                window_len = -1
                for length in range(1, self.cfg.max_window + 1):
                    end = start + length - 1
                    if end > max_ayah:
                        break
                    window_len += len(kb.q_norm_match[ayahs[end]]) + 1
                    len_diff = abs(window_len - query_len)
                    upper_bound = min(1.0, window_len / max(query_len, 1))
                    if best is not None:  
                        best_cov, best_neg = best[0][0], best[0][1]
                        if upper_bound < best_cov or (upper_bound == best_cov and -len_diff < best_neg):
                            continue
                    key_pos = (surah, start, end)
                    if key_pos in memo:
                        continue
                    window = " ".join(kb.q_norm_match[ayahs[a]] for a in range(start, end + 1))
                    matcher = difflib.SequenceMatcher(None, query_norm, window, autojunk=False)
                    matched = sum(b.size for b in matcher.get_matching_blocks() if b.size >= 4)
                    coverage = matched / max(query_len, 1)
                    key = (coverage, -len_diff, matcher.ratio())
                    memo[key_pos] = key
                    if best is None or key > best[0]:
                        best = (key, coverage, key[2], surah, start, end)
        if best is None:
            return None
        _, coverage, ratio, surah, start, end = best
        display = " ".join(kb.quran[kb.quran_by_surah[surah][a]]["text"] for a in range(start, end + 1))
        return CorrectionMatch(
            "Ayah", coverage, ratio, self._ayah_text(surah, start, end),
            {"type": "Quran", "surah_id": surah, "surah_name": kb.quran[kb.quran_by_surah[surah][start]]["surah_name"],
             "ayah_start": start, "ayah_end": end},
            display,
        )

    def _ayah_text(self, surah: int, start: int, end: int) -> str:
        kb, multi = self.kb, end > start
        parts = []
        for a in range(start, end + 1):
            text = kb.quran[kb.quran_by_surah[surah][a]]["text"]
            parts.append(f"{text} ({a})" if multi else text)
        return apply_idgham(" ".join(parts)).replace("\u0640", "")

    def match_hadith(self, query_text: str) -> Optional[CorrectionMatch]:
        kb = self.kb
        query_norm = normalize_for_matching(query_text)
        query_words = content_words(query_norm.split())
        if not query_words:
            return None
        query_len = len(query_norm)
        best = None   
        for idx in kb.hadith_candidates(query_words, self.cfg.hadith_top_k):
            for field_name in ("matn", "full"):
                text = kb.hadith_norm(idx, field_name)
                if not text:
                    continue
                upper_bound = min(1.0, query_len / max(len(text), 1))
                if best is not None and upper_bound < best[0][0]:
                    continue
                matcher = difflib.SequenceMatcher(None, query_norm, text, autojunk=False)
                matched = sum(b.size for b in matcher.get_matching_blocks() if b.size >= 4)
                coverage, candidate_cov = matched / max(query_len, 1), matched / max(len(text), 1)
                key = (min(coverage, candidate_cov), matcher.ratio())
                if best is None or key > best[0]:
                    best = (key, idx, field_name, key[1])
        if best is None:
            return None
        key, idx, field_name, ratio = best
        record = kb.hadith[idx]
        text = record[field_name].strip()
        return CorrectionMatch(
            "Hadith", key[0], ratio, text,
            {"type": "Hadith", "hadithID": record["hadithID"], "book": record["book"], "title": record["title"],
             "field": "matn" if field_name == "matn" else "full_text"},
            text,
        )


STATUS_INFO = {
    "VERIFIED": {"ar": "موثّق", "group": "verified"},
    "CORRECTED": {"ar": "غير مطابق — يوجد تصحيح من المصدر", "group": "mismatch"},
    "UNSUPPORTED": {"ar": "غير مطابق — لا يوجد مصدر مطابق", "group": "mismatch"},
    "HUMAN_REVIEW": {"ar": "يحتاج مراجعة بشرية", "group": "review"},
}


class IslamicContentVerifier:
    

    def __init__(self, retriever: Optional[SourceRetriever] = None, config: Optional[PipelineConfig] = None,
                 use_scanner: bool = True, decouple_triggers: bool = True) -> None:
        self.cfg = config or PipelineConfig()
        self.retriever = retriever or SourceRetriever()
        self.verifier = Verifier(self.retriever, self.cfg.verifier)
        self.corrector = Corrector(self.retriever, self.cfg.corrector)
        self.detector = HybridDetector(self.retriever, use_scanner, rules_use_corpus=decouple_triggers,
                                       decouple_triggers=decouple_triggers)
        self.detector_name = type(self.detector).__name__

    def detect(self, text: str) -> List[DetectedSpan]:
        return self.detector.detect(self._validate(text)) if text.strip() else []

    def needs_hadith(self, text: str) -> bool:
        return any(span.label == "Hadith" for span in self.detect(text))

    def analyze(self, text: str) -> dict:
        text = self._validate(text)
        started = time.time()
        spans = self.detector.detect(text) if text.strip() else []
        detect_seconds = time.time() - started
        result = self._analyze_spans(text, spans)
        result["timings"] = {"detect_s": round(detect_seconds, 3), "total_s": round(time.time() - started, 3)}
        return result

    def analyze_spans(self, text: str, spans: List[dict]) -> dict:
        given = [DetectedSpan(s["start"], s["end"], s["label"], None, "given", text[s["start"]:s["end"]]) for s in spans]
        return self._analyze_spans(self._validate(text), given)

    def analyze_detected(self, text: str, spans: List[DetectedSpan]) -> dict:
        return self._analyze_spans(self._validate(text), spans)

    @staticmethod
    def _validate(text: str) -> str:
        if not isinstance(text, str):
            raise TypeError("Input text must be a string")
        if len(text) > MAX_INPUT_CHARS:
            raise ValueError(f"Input is too long ({len(text)} characters); the limit is {MAX_INPUT_CHARS}")
        return text

    def _analyze_spans(self, text: str, spans: List[DetectedSpan]) -> dict:
        reports = [self._process_span(i + 1, span) for i, span in enumerate(sorted(spans, key=lambda s: s.start))]
        counts = {status: 0 for status in STATUS_INFO}
        for report in reports:
            counts[report["status"]] += 1
        return {
            "input_text": text,
            "detector": self.detector_name,
            "spans": reports,
            "corrected_text": self._apply_corrections(text, reports),
            "summary": {
                "n_spans": len(reports),
                "n_ayah": sum(r["type"] == "Ayah" for r in reports),
                "n_hadith": sum(r["type"] == "Hadith" for r in reports),
                **counts,
                "needs_human_review": counts["HUMAN_REVIEW"] > 0,
            },
        }

    def _process_span(self, index: int, span: DetectedSpan) -> dict:
        try:
            return self._final_verdict(self._ground(self._decide(index, span)))
        except Exception:   
            logger.exception("Failed to process span %d", index)
            report = self._empty_report(index, span)
            self._finalize(report, "HUMAN_REVIEW", {"code": "internal_error"})
            return report

    def _source_text(self, source: Optional[dict]) -> Optional[str]:
        kb = self.retriever
        if not source:
            return None
        if source.get("type") == "Quran":
            ayahs = kb.quran_by_surah.get(source.get("surah_id"), {})
            idxs = [ayahs.get(n) for n in range(source["ayah_start"], source["ayah_end"] + 1)]
            return " ".join(kb.q_norm_match[i] for i in idxs) if idxs and None not in idxs else None
        if source.get("type") == "Hadith":
            if getattr(self, "_hadith_by_id", None) is None:
                self._hadith_by_id = {}
                for record in kb.hadith:
                    self._hadith_by_id.setdefault((record["hadithID"], record["title"]), record)
            record = self._hadith_by_id.get((source.get("hadithID"), source.get("title")))
            if record is not None:
                return normalize_for_matching((record.get("matn") or "") + " " + (record.get("full") or ""))
        return None

    def _ground(self, report: dict) -> dict:
        evidence = report.get("evidence")
        if evidence and self._source_text(evidence["source"]) is None:
            report["evidence"], report["correction"], report["suggestion"] = None, None, None
            self._finalize(report, "HUMAN_REVIEW", {"code": "ungrounded"})
            report["grounding"] = "removed"
            return report
        for key in ("correction", "suggestion"):
            item = report.get(key)
            if not item:
                continue
            source_text = self._source_text(item["source"])
            shown = normalize_for_matching(item["display_text"])
            if source_text is None or shown not in source_text:
                report["correction"] = report["suggestion"] = None
                if report["status"] in ("CORRECTED", "VERIFIED"):
                    self._finalize(report, "HUMAN_REVIEW", {"code": "ungrounded"})
                report["grounding"] = "removed"
                return report
        report["grounding"] = "verified"
        return report

    @staticmethod
    def _final_verdict(report: dict) -> dict:
        report["verification"]["verdict"] = "Correct" if report["status"] == "VERIFIED" else "Incorrect"
        return report

    @staticmethod
    def _empty_report(index: int, span: DetectedSpan) -> dict:
        return {
            "id": index, "type": span.label, "start": span.start, "end": span.end, "text": span.text,
            "detection": {"backend": span.source, "confidence": None if span.confidence is None else round(span.confidence, 4)},
            "verification": {"verdict": "Incorrect", "confidence": 0.0, "score": 0.0, "method": "error", "n_candidates": 0},
            "evidence": None, "correction": None, "suggestion": None, "notes": [],
        }

    @staticmethod
    def _finalize(report: dict, status: str, reason: dict) -> None:
        info = STATUS_INFO[status]
        report.update(status=status, status_ar=info["ar"], group=info["group"], reason=reason)

    @staticmethod
    def source_label(source: dict) -> str:
        if source["type"] == "Quran":
            start, end = source["ayah_start"], source["ayah_end"]
            return f"سورة {source['surah_name']} {start}" + (f"–{end}" if end != start else "")
        return f"حديث رقم {source['hadithID']}"

    def _decide(self, index: int, span: DetectedSpan) -> dict:
        report = self._empty_report(index, span)
        verification = self.verifier.verify(span.text, span.label)
        report["verification"] = {
            "verdict": verification.verdict, "confidence": verification.confidence, "score": verification.best_score,
            "method": verification.method, "n_candidates": verification.n_candidates,
        }
        whole = self._whole_ayah(span.text) if span.label == "Ayah" else None
        if whole is not None:
            self._verified_whole_ayah(report, span, whole)   
        elif span.label == "Ayah":
            self._decide_quran(report, span, verification)
        else:
            self._decide_hadith(report, span, verification)
        if whole is None and span.source == "rules" and len(normalize_for_matching(span.text).split()) < SHORT_QUOTE_WORDS:
            self._finalize(report, "HUMAN_REVIEW", {"code": "too_short"})
            report["correction"] = None
            report["suggestion"] = None
        elif report["status"] in ("UNSUPPORTED", "HUMAN_REVIEW"):
            self._cross_check(report, span)
        if report["status"] == "UNSUPPORTED":
            report["evidence"] = None   
        if report["status"] == "VERIFIED" and span.hint and span.hint != span.label:
            report["notes"].append({"code": "is_ayah" if span.label == "Ayah" else "is_hadith", "source": self.source_label(report["evidence"]["source"]), "misattributed": True})
        return report

    def _whole_ayah(self, text: str) -> Optional[int]:
        if getattr(self, "_whole_map", None) is None:
            mapping: Dict[str, int] = {}
            for i, norm in enumerate(self.retriever.q_norm_match):
                if len(norm.split()) >= 2:
                    mapping.setdefault(norm, i)
            self._whole_map = mapping
        norm = normalize_for_matching(text)
        return self._whole_map.get(norm) if len(norm.split()) >= 2 else None

    def _verified_whole_ayah(self, report: dict, span: DetectedSpan, idx: int) -> None:
        record = self.retriever.quran[idx]
        source_ref = {"type": "Quran", "surah_id": record["surah_id"], "surah_name": record["surah_name"],
                      "ayah_start": record["ayah_id"], "ayah_end": record["ayah_id"]}
        alignment = align(span.text, record["text"], self.retriever.quran_vocabulary)
        report["evidence"] = {"source": source_ref, "signals": compute_signals(span.text, record["text"], "Ayah"),
                              "comparison": alignment}
        report["verification"].update(verdict="Correct", confidence=0.98, score=1.0, method="whole_ayah")
        self._finalize(report, "VERIFIED", {"code": "exact_match", "source": self.source_label(source_ref)})
        if alignment["diacritic_notes"]:
            report["notes"].append({"code": "diacritic_conflict", "n": len(alignment["diacritic_notes"])})

    def _cross_check(self, report: dict, span: DetectedSpan) -> None:
        if len(normalize_for_matching(span.text).split()) < 3:
            return
        try:
            if span.label == "Ayah":
                other = self.verifier._verify_hadith(span.text)
                if other.verdict == "Correct" and other.method == "substring_match" and other.source:
                    report["notes"].append({"code": "is_hadith", "source": f"حديث رقم {other.source['hadithID']}"})
            else:
                other = self.verifier._verify_quran(span.text)
                if other.verdict == "Correct" and other.method == "substring_match" and other.source:
                    c = other.source
                    report["notes"].append({"code": "is_ayah", "source": f"سورة {c['surah_name']} {c['ayah_id']}"})
        except Exception:   
            logger.exception("cross-check failed")

    @staticmethod
    def _proposal(span: DetectedSpan, match: Optional[CorrectionMatch]) -> Optional[dict]:
        if match is None:
            return None
        return {"text": match.text, "display_text": match.display, "source": match.source,
                "match_strength": round(match.strength, 4), "full_ratio": round(match.full_ratio, 4)}

    def _decide_quran(self, report: dict, span: DetectedSpan, verification: Verification) -> None:
        cfg, corr_cfg = self.cfg, self.cfg.corrector
        match = self.corrector.match_quran(span.text)
        if match is not None:
            source_text, source_ref = match.display, match.source
        elif verification.source:
            candidate = verification.source
            source_text = candidate["text"]
            source_ref = {"type": "Quran", "surah_id": candidate["surah_id"], "surah_name": candidate["surah_name"],
                          "ayah_start": candidate["ayah_id"], "ayah_end": candidate["ayah_id"]}
        else:
            source_text, source_ref = "", None
        alignment = align(span.text, source_text, self.retriever.quran_vocabulary) if source_text else None
        if source_ref:
            report["evidence"] = {"source": source_ref, "signals": compute_signals(span.text, source_text, "Ayah"),
                                  "comparison": alignment}
        proposal = self._proposal(span, match)
        n_tokens = len(content_words(normalize_for_matching(span.text).split())) if span.text else 0
        has_tokens = alignment is not None and alignment["exact"] and len(alignment["word_diff"]) >= 1

        if has_tokens and sum(len(op["span"].split()) for op in alignment["word_diff"]) >= MIN_EXACT_TOKENS:
            self._finalize(report, "VERIFIED", {"code": "exact_match", "source": self.source_label(source_ref)})
            if alignment["diacritic_notes"]:
                report["notes"].append({"code": "diacritic_conflict", "n": len(alignment["diacritic_notes"])})
            if alignment["orthographic_variants"]:
                report["notes"].append({"code": "orthographic_variant", "n": alignment["orthographic_variants"]})
            return

        local = bool(alignment and alignment.get("near") and alignment.get("word_similarity", 0) >= 0.75 and alignment.get("source_excerpt"))
        strong = match is not None and match.strength >= corr_cfg.quran_strong and (match.full_ratio >= corr_cfg.min_full_ratio or local)
        if strong:
            self._finalize(report, "CORRECTED", {"code": "altered_passage", "source": self.source_label(source_ref),
                                                 "n": alignment["mismatches"] if alignment else 0,
                                                 "reordered": bool(alignment and alignment["reordered"])})
            excerpt = alignment["source_excerpt"] if alignment and alignment["source_excerpt"] else proposal["display_text"]
            report["correction"] = {**proposal, "display_text": excerpt, "applied": True}
            return

        if verification.verdict == "Correct":
            self._finalize(report, "HUMAN_REVIEW", {"code": "weak_match"})
            report["suggestion"] = proposal
        elif match is None or match.strength < corr_cfg.quran_low:
            confident = verification.confidence >= cfg.unsupported_min_conf and not verification.method.startswith("borderline")
            if match is None or match.strength < cfg.unsupported_strength or confident:
                self._finalize(report, "UNSUPPORTED", {"code": "no_source"})
            else:
                self._finalize(report, "HUMAN_REVIEW", {"code": "insufficient_evidence"})
                report["suggestion"] = proposal
        else:
            self._finalize(report, "HUMAN_REVIEW", {"code": "candidate_not_strong", "source": self.source_label(source_ref),
                                                    "strength": round(match.strength, 2)})
            report["suggestion"] = proposal

    def _decide_hadith(self, report: dict, span: DetectedSpan, verification: Verification) -> None:
        cfg, corr_cfg = self.cfg, self.cfg.corrector
        best = verification.source
        alignment = None
        if best:
            alignment = align(span.text, best["text"])
            report["evidence"] = {
                "source": {"type": "Hadith", "hadithID": best["hadithID"], "book": best["book"], "title": best["title"]},
                "signals": best.get("signals"), "comparison": alignment,
            }
        exact_tokens = alignment is not None and alignment["exact"] and sum(len(op["span"].split()) for op in alignment["word_diff"]) >= 4

        if verification.verdict == "Correct" or exact_tokens:
            altered = (not exact_tokens and alignment is not None and not alignment["exact"] and alignment["mismatches"] >= 1)
            if altered:   
                match = self.corrector.match_hadith(span.text)
                self._finalize(report, "HUMAN_REVIEW", {"code": "hadith_altered", "n": alignment["mismatches"],
                                                        "source": f"حديث رقم {best['hadithID']}"})
                report["suggestion"] = self._proposal(span, match)
                report["verification"]["verdict"] = "Incorrect"
            elif exact_tokens or (not verification.method.startswith("borderline") and verification.confidence >= cfg.verified_min_conf):
                self._finalize(report, "VERIFIED", {"code": "exact_match" if (exact_tokens or (alignment and alignment["exact"])) else "close_match",
                                                    "source": f"حديث رقم {best['hadithID']}"})
                if alignment and not alignment["exact"]:
                    report["notes"].append({"code": "hadith_minor_diffs", "n": alignment["mismatches"]})
                if alignment and alignment["diacritic_notes"]:
                    report["notes"].append({"code": "diacritic_conflict", "n": len(alignment["diacritic_notes"])})
            else:
                self._finalize(report, "HUMAN_REVIEW", {"code": "weak_match"})
            return

        match = self.corrector.match_hadith(span.text)
        proposal = self._proposal(span, match)
        if match is None or match.strength < corr_cfg.hadith_low:
            confident = verification.confidence >= cfg.unsupported_min_conf and not verification.method.startswith("borderline")
            if match is None or match.strength < cfg.unsupported_strength or confident:
                self._finalize(report, "UNSUPPORTED", {"code": "no_source"})
            else:
                self._finalize(report, "HUMAN_REVIEW", {"code": "insufficient_evidence"})
                report["suggestion"] = proposal
        else:   
            self._finalize(report, "HUMAN_REVIEW", {"code": "hadith_candidate", "source": self.source_label(match.source),
                                                    "strength": round(match.strength, 2)})
            report["suggestion"] = proposal

    @staticmethod
    def _apply_corrections(text: str, reports: List[dict]) -> str:
        out = text
        for report in sorted(reports, key=lambda r: r["start"], reverse=True):
            if report["status"] == "CORRECTED" and report["correction"] and report["correction"].get("applied"):
                out = out[: report["start"]] + report["correction"]["display_text"] + out[report["end"]:]
        return out

### Module: `llm_client.py`
Optional LLM client for the Ask & Verify mode, including the foreign-script sanitiser.

In [ ]:
%%writefile llm_client.py

from __future__ import annotations

import json
import os
import re
import urllib.error
import urllib.request
from dataclasses import dataclass
from typing import Dict, Optional, Tuple

SYSTEM_PROMPT = (
    "أنت مساعد معرفي في العلوم الإسلامية. أجب بالعربية بإيجاز ودقة. عند الاستشهاد بآية قرآنية أو حديث نبوي اكتب نصه كاملًا "
    "بين علامتي تنصيص مزدوجتين \"...\" بعد عبارة تمهيدية مثل: قال الله تعالى: أو قال رسول الله ﷺ:. "
    "لا تضع بين علامات التنصيص إلا نص الآية أو الحديث، واذكر السورة ورقم الآية أو مصدر الحديث بعد الاقتباس."
    " اكتب بالحروف العربية فقط ولا تستخدم أي لغة أو كتابة أخرى (لا صينية ولا يابانية ولا إنجليزية). لا تذكر أكثر من حكم أو دليل لا تتأكد منه."
)

DEFAULT_MODEL = "gpt-4o-mini"
DEFAULT_BASE_URL = "https://api.openai.com/v1"
MAX_PROMPT_CHARS = 1500



_FOREIGN = re.compile("[\u0400-\u04ff\u0900-\u097f\u0e00-\u0e7f\u1100-\u11ff\u3000-\u303f\u3040-\u30ff\u3130-\u318f"
                      "\u3400-\u4dbf\u4e00-\u9fff\uac00-\ud7af\uf900-\ufaff\uff00-\uffef]+")


def sanitize_answer(text: str) -> str:
    text = _FOREIGN.sub(" ", text or "")
    text = re.sub(r"([،,؛.])\s*(?:[،,؛])+", r"\1", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r" +([،؛.:])", r"\1", text)
    return text.strip()


class LLMError(RuntimeError):
    """Raised with a user-presentable Arabic message."""


@dataclass
class LLMSettings:
    api_key: str = ""
    model: str = ""
    base_url: str = ""          
    timeout: float = 60.0

    @classmethod
    def from_env(cls) -> "LLMSettings":
        return cls(api_key=os.environ.get("OPENAI_API_KEY", ""), model=os.environ.get("OPENAI_MODEL", ""),
                   base_url=os.environ.get("OPENAI_BASE_URL", ""))


def build_request(settings: LLMSettings, prompt: str) -> Tuple[str, Dict[str, str], dict]:
    base = (settings.base_url or DEFAULT_BASE_URL).rstrip("/")
    body = {"model": settings.model.strip() or DEFAULT_MODEL,
            "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]}
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {settings.api_key}"}
    return f"{base}/chat/completions", headers, body


def parse_response(payload: dict) -> str:
    try:
        text = payload["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise LLMError("وصلت استجابة غير متوقعة من النموذج.") from exc
    if not text or not text.strip():
        raise LLMError("لم يُرجع النموذج أي نص.")
    return sanitize_answer(text)


_HTTP_MESSAGES = {
    400: "رفض المزوّد الطلب.",
    401: "خدمة الإجابة غير مهيّأة بعد (المفتاح غير صالح).",
    403: "الخدمة غير مصرّح لها باستخدام هذا النموذج.",
    404: "النموذج غير متاح لدى المزوّد.",
    429: "تجاوزت حد الاستخدام المسموح؛ حاول لاحقًا.",
}


def generate(settings: Optional[LLMSettings], prompt: str) -> str:
    settings = settings or LLMSettings.from_env()
    if not prompt or not prompt.strip():
        raise LLMError("اكتب سؤالًا أولًا.")
    if len(prompt) > MAX_PROMPT_CHARS:
        raise LLMError(f"السؤال طويل جدًا (الحد الأقصى {MAX_PROMPT_CHARS} حرف).")
    if not settings.api_key or not settings.api_key.strip():
        raise LLMError("خدمة الإجابة غير مهيّأة: لم يُضبط مفتاح الخادم.")
    url, headers, body = build_request(settings, prompt.strip())
    request = urllib.request.Request(url, data=json.dumps(body).encode("utf-8"), headers=headers, method="POST")
    try:
        with urllib.request.urlopen(request, timeout=settings.timeout) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        raise LLMError(_HTTP_MESSAGES.get(exc.code, f"فشل الطلب (الرمز {exc.code}).")) from exc
    except (urllib.error.URLError, TimeoutError) as exc:
        raise LLMError("تعذّر الاتصال بالمزوّد؛ تحقق من الإنترنت.") from exc
    except json.JSONDecodeError as exc:
        raise LLMError("وصلت استجابة غير مقروءة من المزوّد.") from exc
    return parse_response(payload)

### Module: `benchmark_format.py`
IslamicEval 2025 Subtask 1 output formats (1A / 1B / 1C).

In [ ]:
%%writefile benchmark_format.py

from __future__ import annotations

import csv
import io
import json
import re
import sys
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

NO_SOURCE = "خطأ"
FILE_NAMES = ("task1A_predictions.tsv", "task1B_predictions.tsv", "task1C_predictions.tsv")
Row = Tuple


def _clean(value) -> str:
    return str(value).replace("\t", " ").replace("\r", " ").replace("\n", " ").strip()


def competition_rows(result: dict, qid: str) -> Tuple[List[Row], List[Row], List[Row]]:
    a, b, c = [], [], []
    spans = result["spans"]
    if not spans:
        a.append((qid, 0, 0, "No_Spans"))
    for k, span in enumerate(spans, 1):
        a.append((qid, span["start"], span["end"], span["type"]))
        verdict = span["verification"]["verdict"]
        b.append((f"{qid}_{k}", verdict))
        if verdict == "Incorrect":
            correction = span["correction"]["text"] if (span["status"] == "CORRECTED" and span["correction"]) else NO_SOURCE
            c.append((f"{qid}_{span['start']}_{span['end']}", _clean(correction)))
    return a, b, c


def to_tsv(rows: Iterable[Row]) -> str:
    return "".join("\t".join(str(x) for x in row) + "\n" for row in rows)


def official_payload(result: dict, qid: str = "Q1") -> Dict[str, str]:
    return {name: to_tsv(rows) for name, rows in zip(FILE_NAMES, competition_rows(result, qid))}


def write_competition_files(all_rows: Sequence[Sequence[Row]], out_dir: str = "outputs") -> List[Path]:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    paths = []
    for name, rows in zip(FILE_NAMES, all_rows):
        path = out / name
        path.write_text(to_tsv(rows), encoding="utf-8", newline="")
        paths.append(path)
    return paths


def load_responses(xml_path: str) -> Dict[str, str]:
    data = Path(xml_path).read_text(encoding="utf-8")
    out = {}
    for block in re.findall(r"<Question>(.*?)</Question>", data, re.S):
        ident, response = re.search(r"<ID>(.*?)</ID>", block, re.S), re.search(r"<Response>(.*?)</Response>", block, re.S)
        if ident and response:
            out[ident.group(1).strip()] = response.group(1)
    return out


def read_tsv(path: str) -> List[dict]:
    with open(path, encoding="utf-8", newline="") as handle:
        rows = list(csv.reader(handle, delimiter="\t"))
    return [dict(zip(rows[0], row)) for row in rows[1:]] if rows else []


def run_on_texts(pipeline, texts: Dict[str, str], out_dir: str = "outputs") -> Dict[str, dict]:
    acc: Tuple[List[Row], List[Row], List[Row]] = ([], [], [])
    results = {}
    for qid, text in texts.items():
        result = pipeline.analyze(text)
        results[qid] = result
        for target, part in zip(acc, competition_rows(result, qid)):
            target.extend(part)
    write_competition_files(acc, out_dir)
    (Path(out_dir) / "full_report.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    return results


def main(argv: Sequence[str]) -> int:
    if len(argv) < 2:
        print(__doc__)
        return 2
    from verifier import IslamicContentVerifier

    results = run_on_texts(IslamicContentVerifier(), load_responses(argv[1]), argv[2] if len(argv) > 2 else "outputs")
    print(f"{len(results)} responses written to {argv[2] if len(argv) > 2 else 'outputs'}/")
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv))

### Module: `ui.py`
Arabic HTML report renderer (used here to display results inside the notebook).

In [ ]:
%%writefile ui.py

from __future__ import annotations

import html
from typing import List, Optional


_e = html.escape

APP_TITLE = "Amanah AI"
APP_TAGLINE = "نظام ذكي للتحقق من الاقتباسات القرآنية والحديثية"
ENGLISH_TITLE = "AI-Powered Verification and Correction of Quranic and Prophetic Quotations"
COPIED_MESSAGE = "تم نسخ النص بنجاح"

GROUP_LABEL = {"verified": "موثّق", "mismatch": "غير مطابق", "review": "تحتاج مراجعة بشرية"}
TYPE_LABEL = {"Ayah": "آية قرآنية", "Hadith": "حديث نبوي"}
INDICATORS = [
    ("composite", "الدرجة المركبة"),
    ("coverage", "تغطية الكلمات"),
    ("lcs_ratio", "التسلسل النصي"),
    ("token_overlap", "تداخل الكلمات"),
    ("edit_sim", "تشابه الحروف"),
    ("diacritic_sim", "التشكيل"),
]
METHODS = {
    "substring_match": "الاقتباس واردٌ كاملًا في المصدر",
    "threshold_pass": "تجاوز عتبة التطابق",
    "threshold_fail": "دون عتبة التطابق",
    "borderline_multi_cov": "حالة حدّية: عدة مصادر متقاربة",
    "borderline_default": "حالة حدّية",
    "borderline_low_retrieval": "حالة حدّية: استرجاع ضعيف",
    "no_candidates": "لا توجد مصادر مرشحة",
    "empty_span": "نص فارغ",
    "error": "تعذّرت المعالجة",
}


def _pct(value: float) -> str:
    return f"{round(value * 100)}%"


def source_label(source: dict) -> str:
    if source["type"] == "Quran":
        start, end = source["ayah_start"], source["ayah_end"]
        verses = f"{start}" if start == end else f"{start}–{end}"
        return f"سورة {source['surah_name']} — الآية {verses}"
    return f"حديث رقم {source['hadithID']} — {source['title']}"


def reason_text(reason: dict) -> str:
    code = reason["code"]
    src = reason.get("source", "")
    if code == "exact_match":
        return f"تطابق تام مع {src} بعد تجاهل اختلافات الرسم والتشكيل."
    if code == "close_match":
        return f"تطابق شبه كامل مع {src}."
    if code == "altered_passage":
        n = reason.get("n", 0)
        how = "تغيّر ترتيب الكلمات" if reason.get("reordered") else f"{n} موضع مختلف"
        return f"النص يخالف {src} ({how}). التصحيح المقترح هو نص المصدر حرفيًا ولم يُولَّد."
    if code == "weak_match":
        return "وُجد مصدر قريب لكن التطابق غير كافٍ للحكم الآلي."
    if code == "ungrounded":
        return "تعذّر التثبّت من المرجع المقترح داخل المراجع المضمّنة، فأُسقط الاقتراح ولم يُعرض أي تصحيح؛ يلزم الرجوع إلى مختص."
    if code == "too_short":
        return "الاقتباس قصير جدًا (كلمتان فقط) فلا يمكن تحديد مصدره بيقين؛ يلزم الرجوع إلى مختص."
    if code == "no_source":
        return "لا يوجد في المراجع المضمّنة نصٌّ يشبه هذا الاقتباس، وقد يكون مختلَقًا."
    if code == "insufficient_evidence":
        return "الأدلة غير كافية: لا مصدر واضح، ودرجة اليقين منخفضة."
    if code == "candidate_not_strong":
        return f"وُجد مرشح ({src}، قوة المطابقة {round(reason.get('strength', 0) * 100)}%) لكن الدليل لا يكفي للتصحيح الآلي."
    if code == "hadith_altered":
        return (f"النص قريب جدًا من {src} لكن يختلف عنه في {reason.get('n', 1)} موضع (زيادة أو نقص أو استبدال كلمة)، وقد يغيّر ذلك المعنى؛ "
                "لا يُوثَّق الحديث إلا إذا طابق النص حرفيًا، ولا يُصحَّح آليًا فتلزم مراجعة مختص.")
    if code == "hadith_candidate":
        return f"وُجدت رواية مشابهة ({src}) لكن لا يُصحَّح الحديث آليًا لاختلاف الروايات؛ يلزم الرجوع إلى مختص."
    return "تعذّرت معالجة هذا الاقتباس آليًا."


def note_text(note: dict) -> str:
    n = note.get("n", 0)
    code = note["code"]
    if code == "diacritic_conflict":
        return f"تنبيه: تشكيل {n} كلمة يخالف المصحف (الكلمات صحيحة لكن الحركات مختلفة)."
    if code == "orthographic_variant":
        return f"ملاحظة: {n} اختلاف إملائي (رسم أو مسافات) لا يغيّر الكلمة."
    if note.get("misattributed"):
        if code == "is_ayah":
            return f"تنبيه: النص قرآني ({note['source']}) لكن عبارة التقديم تنسبه إلى الحديث."
        return f"تنبيه: النص حديث ({note['source']}) لكن عبارة التقديم تنسبه إلى القرآن."
    if code == "is_hadith":
        return f"تنبيه: هذا النص وارد في الحديث ({note['source']}) وليس في القرآن؛ ربما نُسب إلى الله تعالى خطأً."
    if code == "is_ayah":
        return f"تنبيه: هذا النص وارد في القرآن ({note['source']}) وليس في الحديث؛ ربما نُسب إلى النبي ﷺ خطأً."
    if code == "hadith_minor_diffs":
        return f"ملاحظة: {n} اختلاف طفيف عن أقرب رواية في المراجع."
    return ""


def _diff_html(comparison: dict) -> str:
    """Red = words only in the quotation; green = the source words that should be there."""
    parts = []
    for op in comparison["word_diff"]:
        if op["op"] == "equal":
            parts.append(f'<span class="w-eq">{_e(op["span"])}</span>')
            continue
        if op["span"]:
            parts.append(f'<span class="w-extra">{_e(op["span"])}</span>')
        if op["source"]:
            parts.append(f'<span class="w-missing">{_e(op["source"])}</span>')
    return " ".join(parts)


def _legend(group: str, comparison: Optional[dict]) -> str:
    if group == "verified" or not comparison or all(op["op"] == "equal" for op in comparison["word_diff"]):
        return ""
    return '<p class="legend"><span class="w-extra">كلمات في الاقتباس تخالف المصدر</span> <span class="w-missing">الصواب من المصدر</span></p>'


def _indicator_table(signals: Optional[dict]) -> str:
    if not signals:
        return ""
    rows = []
    for key, label in INDICATORS:
        value = float(signals.get(key, 0.0))
        rows.append(
            f'<div class="ind"><div class="ind-name">{label}</div>'
            f'<div class="bar"><span style="width:{_pct(value)}"></span></div><div class="ind-val">{_pct(value)}</div></div>'
        )
    rows.append(f'<div class="ind"><div class="ind-name">وروده كاملًا في المصدر</div><div class="ind-val wide">{"نعم" if signals.get("is_substring") else "لا"}</div></div>')
    return '<div class="indicators">' + "".join(rows) + "</div>"


def _match_percent(span: dict) -> int:
    evidence = span["evidence"]
    if evidence and evidence.get("comparison"):
        return round(evidence["comparison"]["word_similarity"] * 100)
    if evidence and evidence.get("signals"):
        return round(evidence["signals"]["composite"] * 100)
    return 0


def _evidence_panel(span: dict) -> str:
    evidence, verification = span["evidence"], span["verification"]
    rows = [
        f'<div class="ev-row"><b>الاقتباس المكتشف</b><p class="quote">{_e(span["text"])}</p></div>',
        f'<div class="ev-row"><b>النوع</b><p>{TYPE_LABEL[span["type"]]}</p></div>',
    ]
    if evidence:
        comparison = evidence["comparison"]
        rows += [
            f'<div class="ev-row"><b>المصدر المرشح</b><p>{_e(source_label(evidence["source"]))}</p></div>',
            f'<div class="ev-row"><b>نص المصدر الأصلي</b><p class="quote">{_e(comparison["source_excerpt"])}</p></div>',
            f'<div class="ev-row"><b>المقارنة كلمةً بكلمة</b><p class="diff">{_diff_html(comparison)}</p>{_legend(span["group"], comparison)}</div>',
            f'<div class="ev-row"><b>مؤشرات التحقق</b>{_indicator_table(evidence.get("signals"))}</div>',
        ]
        if comparison["diacritic_notes"]:
            items = "، ".join(f"{_e(n['word'])} ← {_e(n['source_word'])}" for n in comparison["diacritic_notes"])
            rows.append(f'<div class="ev-row"><b>اختلافات التشكيل</b><p>{items}</p></div>')
    else:
        rows.append('<div class="ev-row"><b>المصدر المرشح</b><p>لم يُسترجع أي مصدر.</p></div>')
    rows += [
        f'<div class="ev-row"><b>درجة اليقين في الحكم</b><p>{_pct(verification["confidence"])} · {METHODS.get(verification["method"], "")}</p></div>',
        f'<div class="ev-row"><b>القرار</b><p>{_e(span["status_ar"])}</p></div>',
        f'<div class="ev-row"><b>سبب القرار</b><p>{_e(reason_text(span["reason"]))}</p></div>',
    ]
    return "".join(rows)


def _card(span: dict) -> str:
    group, evidence = span["group"], span["evidence"]
    label = GROUP_LABEL[group]
    chip = f'<span class="conf">نسبة المطابقة <b>{_match_percent(span)}%</b></span>'
    scan_badge = ('<span class="badge scan" title="اكتُشف بمطابقة النص مع المراجع دون علامات تنصيص">اقتباس غير معلَن</span>'
                  if span["detection"]["backend"] == "scan" else "")

    fields = [f'<div class="field"><label>الاقتباس المكتشف</label><p class="quote">{_e(span["text"])}</p></div>']
    if evidence:
        comparison = evidence["comparison"]
        fields.append(f'<div class="field"><label>المصدر المرشح</label><p>{_e(source_label(evidence["source"]))}</p></div>')
        if group != "verified":   
            fields.append(
                f'<div class="field"><label>المقارنة بالمصدر</label><p class="diff">{_diff_html(comparison)}</p>{_legend(group, comparison)}</div>'
            )
    else:
        fields.append('<div class="field"><label>المصدر المرشح</label><p>لا يوجد</p></div>')
    fields.append(f'<p class="reason">{_e(reason_text(span["reason"]))}</p>')

    notes = "".join(f'<p class="note">{_e(note_text(n))}</p>' for n in span["notes"] if note_text(n))
    action = ""
    correction, suggestion = span["correction"], span["suggestion"]
    if correction:
        action = (
            '<div class="action ok"><b>التصحيح المقترح من المصدر</b>'
            f'<p class="quote">{_e(correction["display_text"])}</p>'
            f'<p class="legend">{_e(source_label(correction["source"]))}</p></div>'
        )
    elif group == "review":
        closest = ""
        if suggestion:
            closest = (f'<p class="legend">أقرب مصدر وُجد للمراجِع، وليس تصحيحًا آليًا: {_e(source_label(suggestion["source"]))}</p>'
                       f'<p class="quote small">{_e(suggestion["display_text"][:420])}</p>')
        action = ('<div class="action warn"><b>الأدلة غير كافية للتصحيح الآلي.</b> يُوصى بالمراجعة البشرية.' + closest + "</div>")
    elif span["status"] == "UNSUPPORTED":
        action = '<div class="action bad"><b>لا يوجد مصدر مطابق في المراجع المتاحة.</b> لم يُقترح أي نص بديل.</div>'

    return f"""
<div class="qcard {group}">
  <div class="qhead">
    <span class="idx">{span["id"]}</span>
    <span class="badge type">{TYPE_LABEL[span["type"]]}</span>{scan_badge}
    <span class="badge st {group}">{label}</span>
    {chip}
  </div>
  {"".join(fields)}
  {notes}
  {action}
  <details class="evidence"><summary>عرض الدليل</summary><div class="ev-body">{_evidence_panel(span)}</div></details>
</div>"""


def _summary(summary: dict) -> str:
    tiles = [
        (summary["n_spans"], "إجمالي الاقتباسات", ""),
        (summary["n_ayah"], "آيات قرآنية", ""),
        (summary["n_hadith"], "أحاديث نبوية", ""),
        (summary["VERIFIED"], "موثّق", "verified"),
        (summary["CORRECTED"] + summary["UNSUPPORTED"], "غير مطابق", "mismatch"),
        (summary["HUMAN_REVIEW"], "تحتاج مراجعة", "review"),
    ]
    return '<div class="summary">' + "".join(
        f'<div class="tile {cls}{" zero" if value == 0 and cls else ""}"><b>{value}</b><span>{label}</span></div>'
        for value, label, cls in tiles
    ) + "</div>"


def _highlighted_text(result: dict, title: str) -> str:
    text, pieces, cursor = result["input_text"], [], 0
    for span in result["spans"]:
        pieces.append(_e(text[cursor:span["start"]]))
        pieces.append(f'<mark class="{span["group"]}">{_e(text[span["start"]:span["end"]])}</mark>')
        cursor = span["end"]
    pieces.append(_e(text[cursor:]))
    legend = ('<div class="hl-legend"><mark class="verified">موثّق</mark><mark class="mismatch">غير مطابق</mark>'
              '<mark class="review">مراجعة بشرية</mark></div>')
    return f'<div class="highlight"><label>{title}</label><p>' + "".join(pieces) + "</p>" + legend + "</div>"


def final_text(result: dict) -> str:
    
    text, reports = result["input_text"], result["spans"]
    for report in sorted(reports, key=lambda r: r["start"], reverse=True):
        if report["status"] == "CORRECTED" and report["correction"]:
            text = text[: report["start"]] + report["correction"]["display_text"] + text[report["end"]:]
        elif report["status"] == "UNSUPPORTED":
            text = text[: report["end"]] + " [⚠ لا يوجد مصدر مطابق]" + text[report["end"]:]
        elif report["status"] == "HUMAN_REVIEW":
            text = text[: report["end"]] + " [⚠ يحتاج مراجعة بشرية]" + text[report["end"]:]
    return text


def _final_block(result: dict) -> str:
    summary = result["summary"]
    fixed = summary["CORRECTED"]
    open_items = summary["UNSUPPORTED"] + summary["HUMAN_REVIEW"]
    if fixed == 0 and open_items == 0:
        headline = "كل الاقتباسات المكتشفة مطابقة للمصادر."
    else:
        headline = f"صُحِّح {fixed} اقتباس من نص المصدر، وبقي {open_items} اقتباس يحتاج مراجعة بشرية ومعلَّم بعلامة تنبيه."
    text = final_text(result)
    return (
        '<div class="final"><div class="final-head"><b>النسخة المصحّحة</b>'
        f'<button class="copy" type="button" data-text="{_e(text, quote=True)}" '
        'onclick="window.icvCopy&&window.icvCopy(this)">نسخ النص</button></div>'
        f'<p class="legend">{headline}</p><p class="final-text">{_e(text)}</p></div>'
    )


def render_results(result: dict, generated_answer: Optional[str] = None) -> str:
    header = render_generated_header(generated_answer) if generated_answer is not None else ""
    if not result["spans"]:
        body = (
            '<div class="notice">لم يُعثر على اقتباسات قرآنية أو حديثية في هذا النص. يتعرّف النظام على الاقتباسات بمطابقتها مع المراجع، '
            'سواء وُضعت بين علامات تنصيص أو أقواس أو وردت داخل الكلام دون أي عبارة تمهيدية.</div>'
        )
        return f'<div class="icv">{header}{body}</div>'
    generated = generated_answer is not None
    title = "إجابة النموذج مع الاقتباسات المكتشفة" if generated else "النص مع الاقتباسات المكتشفة"
    parts = [_summary(result["summary"]), _highlighted_text(result, title), "".join(_card(s) for s in result["spans"])]
    summary = result["summary"]
    if generated or summary["CORRECTED"] or summary["UNSUPPORTED"] or summary["HUMAN_REVIEW"]:
        parts.append(_final_block(result))
    return f'<div class="icv">{header}{"".join(parts)}</div>'


def render_generated_header(answer: str) -> str:
    return f'<div class="generated"><label>إجابة النموذج اللغوي كما وصلت</label><p>{_e(answer)}</p></div>'


def render_message(message: str, kind: str = "info") -> str:
    return f'<div class="icv"><div class="notice {kind}">{_e(message)}</div></div>'



STAR = (
    '<svg class="mark" viewBox="0 0 64 64" fill="none" stroke="#E2C06E" stroke-width="1.8" aria-hidden="true">'
    '<rect x="14" y="14" width="36" height="36"/><rect x="14" y="14" width="36" height="36" transform="rotate(45 32 32)"/>'
    '<circle cx="32" cy="32" r="7" fill="#E2C06E" stroke="none"/></svg>'
)

HERO = f"""
<div class="icv"><header class="hero">
  {STAR}
  <h1 class="brand">{APP_TITLE}</h1>
  <p class="tagline">{APP_TAGLINE}</p>
  <p class="sub">يكتشف الاقتباسات القرآنية والحديثية داخل أي نص، ويطابقها كلمةً بكلمة مع نصوص المراجع، ويصحّح الخطأ من المصدر نفسه.
     وحين لا يجد دليلًا كافيًا لا يخمّن أبدًا، بل يعلن أن الاقتباس غير مطابق أو يحيله إلى المراجعة البشرية.</p>
  <ul class="trust"><li><b>من المصدر</b> كل مرجع وتصحيح من نص المراجع حرفيًا</li><li><b>٣ قرارات</b> موثّق · غير مطابق · مراجعة بشرية</li><li><b>داخل المتصفح</b> التحقق لا يرسل نصك إلى أي خادم</li></ul>
</header></div>
"""

DISCLAIMER = """<div class="icv"><div class="disclaimer">أداة مساعدة للتدقيق النصي وليست فتوى ولا بديلًا عن المراجعة المتخصصة.
النتائج مبنية على مراجع القرآن الكريم والكتب الستة المضمّنة فقط.
<br>نص القرآن: <a href="https://tanzil.net" target="_blank" rel="noopener">مشروع <bdi>Tanzil</bdi></a> <bdi>(CC BY 3.0)</bdi> · بيانات <bdi>IslamicEval 2025</bdi></div></div>"""

PLACEHOLDER = "الصق هنا النص الذي ولّده نموذج لغوي. يكتشف النظام الآيات والأحاديث الواردة فيه، بعلامات تنصيص أو بدونها…"
PROMPT_PLACEHOLDER = "اكتب سؤالك، مثل: اشرح لي فضل الصبر في القرآن والسنة مع ذكر الأدلة."

_PATTERN = (
    "url(\"data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='88' height='88' viewBox='0 0 88 88'%3E"
    "%3Cg fill='none' stroke='%230F4C3A' stroke-opacity='0.07' stroke-width='1'%3E"
    "%3Crect x='22' y='22' width='44' height='44'/%3E%3Crect x='22' y='22' width='44' height='44' transform='rotate(45 44 44)'/%3E"
    "%3C/g%3E%3C/svg%3E\")"
)

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Cairo:wght@400;600;700&family=Amiri:wght@400;700&display=swap');
:root { --green:#0F4C3A; --green-2:#17694F; --gold:#B8912F; --gold-2:#E2C06E; --cream:#FBF6EA; --paper:#FFFFFF;
        --ink:#1F2933; --muted:#55626D; --line:#E6DCC3; --ok:#1B7A4B; --ok-bg:#EAF6EF; --ok-line:#BFE3CD;
        --bad:#B3261E; --bad-bg:#FDECEA; --bad-line:#F4B8B3; --warn:#7A5B0C; --warn-bg:#FFF6DA; --warn-line:#EBD28A; }
.gradio-container, body.icv-page { background:var(--cream) PATTERN !important; font-family:'Cairo','Segoe UI',Tahoma,sans-serif; color:var(--ink); }
.gradio-container { max-width:1060px !important; --body-text-color:#1F2933; --block-background-fill:#FFFFFF; --block-border-color:#E6DCC3;
  --input-background-fill:#FFFFFF; --block-label-text-color:#55626D; --button-primary-background-fill:#0F4C3A;
  --button-primary-background-fill-hover:#17694F; --button-primary-text-color:#FFFFFF; --button-secondary-background-fill:#FFFFFF;
  --button-secondary-text-color:#0F4C3A; --button-secondary-border-color:#B8912F; --color-accent:#B8912F; }
.icv { direction:rtl; text-align:right; color:var(--ink); line-height:1.9; }
.icv .hero { background:linear-gradient(135deg,#0F4C3A,#17694F); border-radius:20px; padding:30px 22px 26px; margin:10px 0 18px; text-align:center;
  box-shadow:0 6px 22px rgba(15,76,58,.18); border-bottom:4px solid var(--gold); }
.icv .hero .mark { width:54px; height:54px; display:block; margin:0 auto 4px; }
.icv .hero h1 { font-size:2.3rem; margin:.1rem 0; color:#FFFFFF; font-weight:700; }
.icv .hero .tagline { color:var(--gold-2); font-size:1.2rem; font-weight:600; margin:.1rem 0 .6rem; }
.icv .hero .sub { max-width:720px; margin:0 auto; color:#E9F2EE; font-size:1rem; }
.icv .flow { display:flex; flex-wrap:wrap; justify-content:center; align-items:center; gap:8px; margin-top:16px; }
.icv .flow span { background:rgba(255,255,255,.12); border:1px solid rgba(226,192,110,.55); color:#FFF; border-radius:999px; padding:3px 16px; font-size:.9rem; }
.icv .flow i { color:var(--gold-2); font-style:normal; font-size:1.2rem; }
.icv .disclaimer { font-size:.85rem; color:var(--muted); text-align:center; padding:14px 8px; }
.icv .en-free { direction:rtl; }
.input-area textarea { direction:rtl; text-align:right; font-family:'Amiri','Cairo',serif !important; font-size:1.25rem !important; line-height:2.1 !important; background:#FFFFFF !important; color:#1F2933 !important; }
.llm-row label, .gradio-container label span { font-family:'Cairo',sans-serif; }
.results { direction:rtl; }
.icv .summary { display:grid; grid-template-columns:repeat(auto-fit,minmax(130px,1fr)); gap:10px; margin:8px 0 14px; }
.icv .tile { background:var(--paper); border:1px solid var(--line); border-radius:14px; padding:12px 8px; text-align:center; }
.icv .tile b { display:block; font-size:1.8rem; color:var(--green); line-height:1.3; } .icv .tile span { font-size:.92rem; color:var(--muted); }
.icv .tile.verified { background:var(--ok-bg); border-color:var(--ok-line); } .icv .tile.verified b { color:var(--ok); }
.icv .tile.mismatch { background:var(--bad-bg); border-color:var(--bad-line); } .icv .tile.mismatch b { color:var(--bad); }
.icv .tile.review { background:var(--warn-bg); border-color:var(--warn-line); } .icv .tile.review b { color:var(--warn); }
.icv .tile.zero { background:var(--paper); border-color:var(--line); opacity:.6; } .icv .tile.zero b { color:var(--muted); }
.icv .highlight, .icv .generated, .icv .final { background:var(--paper); border:1px solid var(--line); border-radius:14px; padding:14px 18px; margin-bottom:14px; }
.icv label { display:block; color:var(--muted); font-size:.82rem; font-weight:600; margin-bottom:4px; }
.icv .highlight p, .icv .generated p, .icv .final-text { font-family:'Amiri','Cairo',serif; font-size:1.2rem; line-height:2.2; margin:0; white-space:pre-wrap; }
.icv mark { color:var(--ink); border-radius:6px; padding:1px 5px; }
.icv mark.verified { background:#D5EEDF; } .icv mark.mismatch { background:#F8CFCB; } .icv mark.review { background:#F7E3A6; }
.icv .hl-legend { display:flex; flex-wrap:wrap; gap:8px; margin-top:10px; font-size:.78rem; }
.icv .qcard { background:var(--paper); border:1px solid var(--line); border-inline-start:6px solid var(--line); border-radius:16px; padding:16px 18px; margin:12px 0; }
.icv .qcard.verified { border-inline-start-color:var(--ok); } .icv .qcard.mismatch { border-inline-start-color:var(--bad); } .icv .qcard.review { border-inline-start-color:var(--gold); }
.icv .qhead { display:flex; flex-wrap:wrap; gap:8px; align-items:center; margin-bottom:8px; }
.icv .idx { background:var(--green); color:#FFF; border-radius:50%; width:28px; height:28px; display:inline-flex; align-items:center; justify-content:center; font-weight:700; font-size:.9rem; }
.icv .badge { padding:2px 14px; border-radius:999px; font-size:.84rem; background:#F3EEDD; border:1px solid var(--line); color:var(--ink); }
.icv .badge.st.verified { background:var(--ok-bg); border-color:var(--ok-line); color:var(--ok); font-weight:600; }
.icv .badge.st.mismatch { background:var(--bad-bg); border-color:var(--bad-line); color:var(--bad); font-weight:600; }
.icv .badge.st.review { background:var(--warn-bg); border-color:var(--warn-line); color:var(--warn); font-weight:600; }
.icv .badge.scan { background:#EEF3FB; border-color:#C9D8EE; color:#2F4F7F; }
.icv .conf { margin-inline-start:auto; color:var(--muted); font-size:.88rem; } .icv .conf b { color:var(--green); }
.icv .field { margin:8px 0; } .icv .field p { margin:0; }
.icv .quote { font-family:'Amiri','Cairo',serif; font-size:1.25rem; line-height:2.2; color:var(--ink); } .icv .quote.small { font-size:1.05rem; color:#3a4651; }
.icv .diff { font-family:'Amiri','Cairo',serif; font-size:1.2rem; line-height:2.2; }
.icv .w-extra { background:var(--bad-bg); color:var(--bad); border-radius:4px; padding:0 4px; text-decoration:line-through; }
.icv .w-missing { background:var(--ok-bg); color:var(--ok); border-radius:4px; padding:0 4px; font-weight:700; }
.icv .legend { color:var(--muted); font-size:.82rem; margin:4px 0 0; } .icv .legend span { text-decoration:none; font-size:.8rem; }
.icv .reason { color:var(--muted); font-size:.92rem; margin:6px 0 0; }
.icv .note { color:var(--warn); background:var(--warn-bg); border:1px solid var(--warn-line); border-radius:10px; padding:5px 12px; font-size:.88rem; margin:8px 0 0; }
.icv .action { border-radius:12px; padding:10px 14px; margin-top:10px; }
.icv .action.ok { background:var(--ok-bg); border:1px solid var(--ok-line); } .icv .action.ok b { color:var(--ok); }
.icv .action.warn { background:var(--warn-bg); border:1px solid var(--warn-line); } .icv .action.warn b { color:var(--warn); }
.icv .action.bad { background:var(--bad-bg); border:1px solid var(--bad-line); } .icv .action.bad b { color:var(--bad); }
.icv .evidence { margin-top:12px; border-top:1px dashed var(--line); padding-top:8px; }
.icv .evidence summary { cursor:pointer; color:var(--green); font-weight:700; }
.icv .ev-row { margin:10px 0; } .icv .ev-row b { color:var(--gold); font-size:.88rem; } .icv .ev-row p { margin:2px 0; }
.icv .indicators { display:grid; gap:6px; margin-top:6px; }
.icv .ind { display:grid; grid-template-columns:150px 1fr 48px; gap:10px; align-items:center; font-size:.88rem; }
.icv .bar { background:#EFE8D3; border-radius:999px; height:9px; overflow:hidden; direction:rtl; } .icv .bar span { display:block; height:100%; background:linear-gradient(270deg,var(--green),var(--gold)); }
.icv .ind-val { text-align:left; direction:ltr; color:var(--ink); } .icv .ind-val.wide { grid-column:2 / span 2; text-align:right; direction:rtl; }
.icv .final { border-color:var(--gold); background:#FFFDF6; }
.icv .final-head { display:flex; justify-content:space-between; align-items:center; } .icv .final-head b { color:var(--green); font-size:1.05rem; }
.icv .copy { background:var(--green); color:#FFF; border:0; border-radius:10px; padding:5px 16px; font-family:inherit; cursor:pointer; }
.icv .notice { background:var(--paper); border:1px solid var(--line); border-radius:14px; padding:16px; }
.icv .notice.warn { background:var(--warn-bg); border-color:var(--warn-line); color:var(--warn); } .icv .notice.bad { background:var(--bad-bg); border-color:var(--bad-line); color:var(--bad); }
@media (max-width:640px){ .icv .hero h1{font-size:1.7rem;} .icv .ind{grid-template-columns:104px 1fr 40px;} }
""".replace("PATTERN", _PATTERN)


GLASS_CSS = """
:root { --glass:rgba(255,255,255,.62); --glass-strong:rgba(255,255,255,.82); --glass-line:rgba(255,255,255,.75);
        --shadow-1:0 1px 2px rgba(15,76,58,.06), 0 8px 24px rgba(15,76,58,.08); --shadow-2:0 2px 4px rgba(15,76,58,.08), 0 18px 44px rgba(15,76,58,.16); }
.gradio-container, body.icv-page { background:
    radial-gradient(900px 520px at 88% -8%, rgba(226,192,110,.34), transparent 60%),
    radial-gradient(760px 520px at 6% 4%, rgba(23,105,79,.20), transparent 62%),
    linear-gradient(180deg,#FBF6EA 0%,#F3EBD3 100%) !important; background-attachment:fixed !important; }
/* RTL safety: long words, URLs and mixed Latin/digit runs wrap inside their box instead of spilling out */
.icv, .icv * { box-sizing:border-box; min-width:0; }
.icv p, .icv span, .icv b, .icv label, .icv summary, .icv button, .icv .badge, .icv .tile { overflow-wrap:anywhere; }
.icv .quote, .icv .diff, .icv .final-text, .icv .highlight p, .icv .generated p { unicode-bidi:plaintext; text-align:start; }
.icv .hero { position:relative; overflow:hidden; border-bottom:0; border:1px solid rgba(226,192,110,.45);
  background:linear-gradient(120deg,#0B3B2D,#17694F 45%,#0F4C3A 70%,#1d7a5c); background-size:240% 240%; animation:icv-flow 16s ease-in-out infinite;
  box-shadow:var(--shadow-2); }
.icv .hero::before { content:""; position:absolute; inset:-40% -10% auto auto; width:60%; aspect-ratio:1; border-radius:50%;
  background:radial-gradient(circle, rgba(226,192,110,.38), transparent 65%); pointer-events:none; }
.icv .hero::after { content:""; position:absolute; inset:auto auto 0 0; width:100%; height:4px; background:linear-gradient(90deg,transparent,var(--gold-2),transparent); }
.icv .hero h1 { font-size:clamp(1.45rem,4.2vw,2.3rem); line-height:1.5; text-wrap:balance; position:relative; }
.icv .hero .tagline, .icv .hero .sub, .icv .flow { position:relative; }
.icv .flow span { backdrop-filter:blur(6px); -webkit-backdrop-filter:blur(6px); }
@keyframes icv-flow { 0%,100%{background-position:0% 50%} 50%{background-position:100% 50%} }
.icv .tile, .icv .highlight, .icv .generated, .icv .final, .icv .qcard, .icv .notice, .icv .export {
  background:var(--glass); border:1px solid var(--glass-line); box-shadow:var(--shadow-1);
  backdrop-filter:blur(14px) saturate(150%); -webkit-backdrop-filter:blur(14px) saturate(150%); transition:transform .25s ease, box-shadow .25s ease; }
.icv .qcard { border-inline-start:6px solid var(--line); }
.icv .qcard:hover, .icv .tile:hover { transform:translateY(-2px); box-shadow:var(--shadow-2); }
.icv .tile.verified { background:linear-gradient(160deg,rgba(234,246,239,.92),rgba(255,255,255,.6)); }
.icv .tile.mismatch { background:linear-gradient(160deg,rgba(253,236,234,.92),rgba(255,255,255,.6)); }
.icv .tile.review { background:linear-gradient(160deg,rgba(255,246,218,.95),rgba(255,255,255,.6)); }
.icv .final { background:linear-gradient(160deg,rgba(255,253,246,.92),rgba(250,240,208,.55)); border-color:rgba(184,145,47,.55); }
.icv .final-head { flex-wrap:wrap; gap:10px; }
.icv .qhead .badge { max-width:100%; white-space:normal; }
.icv .ind { grid-template-columns:minmax(96px,150px) minmax(0,1fr) 48px; }
.icv .bar span { background:linear-gradient(270deg,var(--green),var(--gold-2)); transition:width .6s ease; }
.icv .copy, .icv .dl { background:linear-gradient(135deg,var(--green),var(--green-2)); color:#FFF; border:0; border-radius:12px; padding:7px 18px;
  font-family:inherit; font-weight:600; cursor:pointer; box-shadow:0 4px 12px rgba(15,76,58,.25); transition:transform .15s ease, box-shadow .15s ease, filter .15s ease; }
.icv .copy:hover, .icv .dl:hover { transform:translateY(-1px); filter:brightness(1.08); box-shadow:0 8px 18px rgba(15,76,58,.3); }
.icv .copy:active, .icv .dl:active { transform:translateY(0); }
.icv .copy:focus-visible, .icv .dl:focus-visible { outline:3px solid rgba(184,145,47,.55); outline-offset:2px; }
.icv .copy.done { background:linear-gradient(135deg,#1B7A4B,#2a9d66); }
.icv .export { border-radius:16px; padding:12px 18px; margin:12px 0; }
.icv .export summary { cursor:pointer; color:var(--green); font-weight:700; }
.icv .dls { display:flex; flex-wrap:wrap; gap:10px; margin-top:10px; }
/* toast */
#icv-toast { position:fixed; inset-inline:0; bottom:28px; margin-inline:auto; width:max-content; max-width:calc(100vw - 32px); z-index:99999;
  direction:rtl; text-align:center; font:600 1rem 'Cairo','Segoe UI',Tahoma,sans-serif; color:#FFF; padding:12px 24px; border-radius:999px;
  background:linear-gradient(135deg,rgba(15,76,58,.94),rgba(23,105,79,.94)); border:1px solid rgba(226,192,110,.6);
  box-shadow:0 14px 40px rgba(15,76,58,.35); backdrop-filter:blur(10px); -webkit-backdrop-filter:blur(10px);
  opacity:0; transform:translateY(18px) scale(.97); pointer-events:none; transition:opacity .28s ease, transform .28s cubic-bezier(.2,.9,.3,1.2); }
#icv-toast.show { opacity:1; transform:none; } #icv-toast.bad { background:linear-gradient(135deg,rgba(179,38,30,.95),rgba(214,69,58,.95)); }
/* skeleton loaders */
.skel { position:relative; overflow:hidden; background:rgba(15,76,58,.08); border-radius:10px; }
.skel::after { content:""; position:absolute; inset:0; transform:translateX(100%); animation:icv-shimmer 1.4s infinite;
  background:linear-gradient(90deg,transparent,rgba(255,255,255,.75),transparent); }
@keyframes icv-shimmer { 100% { transform:translateX(-100%); } }
.skel-card { background:var(--glass); border:1px solid var(--glass-line); border-radius:16px; padding:16px 18px; margin:12px 0; box-shadow:var(--shadow-1); }
.skel-line { height:14px; margin:10px 0; } .skel-line.w60 { width:60%; } .skel-line.w85 { width:85%; } .skel-line.w40 { width:40%; }
.skel-tiles { display:grid; grid-template-columns:repeat(auto-fit,minmax(120px,1fr)); gap:10px; margin:8px 0 14px; } .skel-tile { height:78px; border-radius:14px; }
.icv-spinner { width:18px; height:18px; border-radius:50%; border:3px solid rgba(15,76,58,.18); border-top-color:var(--gold); display:inline-block;
  vertical-align:middle; margin-inline-end:10px; animation:icv-spin .8s linear infinite; }
@keyframes icv-spin { to { transform:rotate(360deg); } }
@media (prefers-reduced-motion: reduce) { .icv .hero, .skel::after, .icv-spinner { animation:none; } .icv .qcard, .icv .tile, #icv-toast { transition:none; } }
@media (max-width:640px){ .icv .ind{grid-template-columns:minmax(84px,104px) minmax(0,1fr) 40px;} .icv .final-head{flex-direction:column; align-items:stretch;} .icv .copy{width:100%;} }
/* layout polish: centred, compact, professional */
.icv .notice, .icv .disclaimer, .icv .export { text-align:center; }
.icv .export .legend { max-width:640px; margin:8px auto 0; font-size:.9rem; line-height:1.9; color:var(--muted); }
.icv .dls { justify-content:center; }
.icv .dl { display:inline-flex; flex-direction:column; align-items:center; gap:2px; min-width:170px; }
.icv .dl .t { font-weight:700; } .icv .dl .s { font-size:.78rem; font-weight:500; opacity:.85; }
.icv .hero .flow { display:flex; justify-content:center; flex-wrap:wrap; gap:8px; }

/* ---- Amanah AI visual identity ---- */
.icv .hero { padding:38px 24px 30px; border-radius:26px; background:
  radial-gradient(900px 300px at 85% -10%, rgba(226,192,110,.28), transparent 60%),
  radial-gradient(700px 320px at 0% 110%, rgba(23,105,79,.9), transparent 70%),
  linear-gradient(135deg,#0A3B2C 0%,#0F4C3A 45%,#17694F 100%); box-shadow:0 18px 50px rgba(10,59,44,.28); }
.icv .hero .mark { width:60px; height:60px; margin-bottom:6px; filter:drop-shadow(0 4px 10px rgba(226,192,110,.45)); }
.icv .hero h1.brand { font-family:'Cairo','Segoe UI',sans-serif; font-weight:800; font-size:clamp(2.4rem,7vw,3.9rem); letter-spacing:.02em; direction:ltr;
  line-height:1.15; margin:.1rem 0 .2rem; background:linear-gradient(180deg,#FFFFFF 30%,#F3DFA6 100%); -webkit-background-clip:text; background-clip:text; color:transparent; }
.icv .hero .tagline { font-size:clamp(1.05rem,2.6vw,1.45rem); font-weight:700; color:var(--gold-2); margin:.2rem 0 .8rem; }
.icv .hero .sub { max-width:760px; font-size:1.02rem; line-height:2; color:#E4EFEA; }
.icv .trust { list-style:none; display:flex; flex-wrap:wrap; justify-content:center; gap:10px; margin:20px 0 0; padding:0; position:relative; }
.icv .trust li { background:rgba(255,255,255,.1); border:1px solid rgba(226,192,110,.5); border-radius:14px; padding:8px 16px; color:#FFF; font-size:.92rem; backdrop-filter:blur(6px); }
.icv .trust li b { color:var(--gold-2); font-size:1.05rem; margin-inline-end:6px; }
.icv .tile { box-shadow:0 6px 18px rgba(15,76,58,.07); transition:transform .2s; } .icv .tile:hover { transform:translateY(-2px); }
.icv .qcard { box-shadow:0 8px 26px rgba(15,76,58,.09); }
"""
CSS += GLASS_CSS

SKELETON = (
    '<div class="icv" aria-busy="true" aria-label="جارٍ التحقق">'
    '<div class="skel-tiles">' + '<div class="skel skel-tile"></div>' * 4 + '</div>'
    + ('<div class="skel-card"><div class="skel skel-line w40"></div><div class="skel skel-line w85"></div>'
       '<div class="skel skel-line"></div><div class="skel skel-line w60"></div></div>') * 2
    + '</div>'
)

COPY_JS = r"""
(function () {
  if (window.icvCopy) return;
  function toast(message, bad) {
    var t = document.getElementById('icv-toast');
    if (!t) { t = document.createElement('div'); t.id = 'icv-toast'; t.setAttribute('role', 'status'); t.setAttribute('aria-live', 'polite'); document.body.appendChild(t); }
    t.textContent = message; t.className = 'show' + (bad ? ' bad' : '');
    clearTimeout(window.__icvToast); window.__icvToast = setTimeout(function () { t.className = bad ? 'bad' : ''; }, 2400);
  }
  function legacyCopy(text) {
    var area = document.createElement('textarea'); area.value = text; area.setAttribute('readonly', '');
    area.style.cssText = 'position:fixed;top:0;left:0;opacity:0;pointer-events:none'; document.body.appendChild(area);
    area.select(); area.setSelectionRange(0, text.length); var ok = false;
    try { ok = document.execCommand('copy'); } catch (e) { ok = false; }
    document.body.removeChild(area); return ok;
  }
  window.icvToast = toast;
  window.icvCopy = function (button) {
    var text = button.getAttribute('data-text') || '';
    function finish(ok) {
      toast(ok ? '__COPIED__' : 'تعذّر النسخ، حدّد النص وانسخه يدويًا', !ok);
      if (ok) { button.classList.add('done'); setTimeout(function () { button.classList.remove('done'); }, 1600); }
    }
    if (navigator.clipboard && window.isSecureContext) {
      navigator.clipboard.writeText(text).then(function () { finish(true); }, function () { finish(legacyCopy(text)); });
    } else { finish(legacyCopy(text)); }
  };
  window.icvDownload = function (button) {
    var blob = new Blob([button.getAttribute('data-text') || ''], { type: 'text/tab-separated-values;charset=utf-8' });
    var link = document.createElement('a'); link.href = URL.createObjectURL(blob); link.download = button.getAttribute('data-name') || 'results.tsv';
    document.body.appendChild(link); link.click(); document.body.removeChild(link); setTimeout(function () { URL.revokeObjectURL(link.href); }, 1000);
  };
})();
""".replace("__COPIED__", COPIED_MESSAGE)

## 3. Build the indexes if needed
Only runs when the JSON corpora were provided instead of the prebuilt indexes.

In [ ]:
if not all((WORKDIR / "index" / n).exists() for n in ("quran.idx.gz", "hadith.idx.gz")):
    from index_builder import build_quran_index, build_hadith_index, save_index
    save_index(build_quran_index(WORKDIR / "data" / "quran.json"), WORKDIR / "index" / "quran.idx.gz")
    save_index(build_hadith_index(WORKDIR / "data" / "hadith.json"), WORKDIR / "index" / "hadith.idx.gz")
print("Indexes ready:", sorted(p.name for p in (WORKDIR / "index").glob("*")))

## 4. Load the pipeline
Creates the verifier (Quran index first, Hadith index on first use) and warms the Hadith index.

In [ ]:
import time
from verifier import IslamicContentVerifier

started = time.time()
pipeline = IslamicContentVerifier()
pipeline.retriever.warm()
print(f"Pipeline ready in {time.time() - started:.1f}s")

## 5. Verify a text
Paste any Arabic text. Each Quran/Hadith quotation is detected, compared word by word with its source, and marked *verified*, *incorrect* (with the exact source text as correction) or *needs human review*.

In [ ]:
from IPython.display import HTML, display
import ui

def verify(text, show_html=True):
    result = pipeline.analyze(text)
    if show_html:
        display(HTML("<style>" + ui.CSS + "</style>" + ui.render_results(result)))
    return result

sample = 'قال الله تعالى: "إِنَّ مَعَ الْعُسْرِ يُسْرًا". وقال رسول الله ﷺ: "الطُّهُورُ شَطْرُ الْإِيمَانِ" رواه مسلم.'
result = verify(sample)

## 6. Compact decision table
A plain-text view of every decision, useful for logs and for checking the abstention behaviour.

In [ ]:
def table(result):
    rows = [(s["id"], s["type"], s["status"], s["reason"]["code"], s["grounding"], s["text"][:48]) for s in result["spans"]]
    print(f"{'#':>2}  {'type':6} {'status':13} {'reason':22} {'grounding':10} text")
    for r in rows:
        print(f"{r[0]:>2}  {r[1]:6} {r[2]:13} {r[3]:22} {r[4]:10} {r[5]}")

table(result)

## 7. Run the bundled examples
The examples cover verified, incorrect (with correction), unsupported, uncertain and human-review cases. The next cell writes them to `demo/examples.json`.

In [ ]:
%%writefile demo/examples.json
[
  {
    "id": "scenario_all_correct_ayahs",
    "title": "كل الآيات صحيحة",
    "question": "ما فضل الصبر واليسر بعد العسر وغاية خلق الإنسان في القرآن؟",
    "scenario": "سيناريو للتجربة: كل الآيات صحيحة",
    "text": "ذكر القرآن حكمة خلق الإنسان وبشّر بعد الشدة بالفرج. قال الله تعالى: \"وَمَا خَلَقْتُ الْجِنَّ وَالْإِنسَ إِلَّا لِيَعْبُدُونِ\" [الذاريات: 56]. وقال سبحانه: \"فَإِنَّ مَعَ الْعُسْرِ يُسْرًا ۝ إِنَّ مَعَ الْعُسْرِ يُسْرًا\" [الشرح: 5-6]. وقال عز وجل: \"وَاسْتَعِينُوا بِالصَّبْرِ وَالصَّلَاةِ ۚ وَإِنَّهَا لَكَبِيرَةٌ إِلَّا عَلَى الْخَاشِعِينَ\" [البقرة: 45].",
    "built_from": "كُتب من نصوص القرآن والحديث المضمّنة؛ الأخطاء مصطنعة لغرض التجربة",
    "observed_statuses": [
      "VERIFIED",
      "VERIFIED",
      "VERIFIED"
    ]
  },
  {
    "id": "scenario_all_wrong_ayahs",
    "title": "كل الآيات خاطئة",
    "question": "ما الآيات الواردة في العبادة والفرج والصبر؟",
    "scenario": "سيناريو للتجربة: كل الآيات خاطئة",
    "text": "وردت في القرآن آيات في هذا الباب. قال الله تعالى: \"وَمَا خَلَقْتُ الْجِنَّ وَالْإِنسَ إِلَّا لِيَأْكُلُونِ\". وقال تعالى: \"فَإِنَّ مَعَ الْعُسْرِ يُسْرًا ۝ إِنَّ بَعْدَ الْعُسْرِ سُرُورًا\". وقال سبحانه: \"وَاسْتَعِينُوا بِالصَّبْرِ وَالصَّلَاةِ ۚ وَإِنَّهَا لَسَهْلَةٌ عَلَى الْخَاشِعِينَ\".",
    "built_from": "كُتب من نصوص القرآن والحديث المضمّنة؛ الأخطاء مصطنعة لغرض التجربة",
    "observed_statuses": [
      "CORRECTED",
      "CORRECTED",
      "CORRECTED"
    ]
  },
  {
    "id": "scenario_all_correct_hadiths",
    "title": "كل الأحاديث صحيحة",
    "question": "ما الأحاديث الصحيحة في النية والطهارة وحفظ اللسان؟",
    "scenario": "سيناريو للتجربة: كل الأحاديث صحيحة",
    "text": "من الأحاديث الصحيحة في هذا الباب: قال رسول الله ﷺ: \"إِنَّمَا الْأَعْمَالُ بِالنِّيَّاتِ ، وَإِنَّمَا لِكُلِّ امْرِئٍ مَا نَوَى\" رواه البخاري. وقال ﷺ: \"الطُّهُورُ شَطْرُ الْإِيمَانِ\" رواه مسلم. وقال ﷺ: \"مَنْ كَانَ يُؤْمِنُ بِاللَّهِ وَالْيَوْمِ الْآخِرِ فَلْيَقُلْ خَيْرًا أَوْ لِيَصْمُتْ\" متفق عليه.",
    "built_from": "كُتب من نصوص القرآن والحديث المضمّنة؛ الأخطاء مصطنعة لغرض التجربة",
    "observed_statuses": [
      "VERIFIED",
      "VERIFIED",
      "VERIFIED"
    ]
  },
  {
    "id": "scenario_all_wrong_hadiths",
    "title": "كل الأحاديث خاطئة",
    "question": "هل ورد حديث في فضل الذكر والصدقة وبر الوالدين؟",
    "scenario": "سيناريو للتجربة: كل الأحاديث خاطئة",
    "text": "يُروى في هذا الباب: قال رسول الله ﷺ: \"مَنْ لَبِسَ الْأَخْضَرَ يَوْمَ الْجُمُعَةِ زَادَ اللَّهُ فِي رِزْقِهِ عَشْرَةَ أَضْعَافٍ\". وقال ﷺ: \"مَنْ أَكَلَ الْعَسَلَ مَعَ الثُّومِ سَبْعَةَ أَيَّامٍ شُفِيَ مِنْ كُلِّ دَاءٍ بِإِذْنِ اللَّهِ\". وقال ﷺ: \"مَنْ دَخَلَ بَيْتَهُ بِرِجْلِهِ الْيُسْرَى انْتَقَصَ عُمُرُهُ سَنَةً كَامِلَةً\".",
    "built_from": "كُتب من نصوص القرآن والحديث المضمّنة؛ الأخطاء مصطنعة لغرض التجربة",
    "observed_statuses": [
      "UNSUPPORTED",
      "UNSUPPORTED",
      "UNSUPPORTED"
    ]
  },
  {
    "id": "scenario_wrong_ayahs_correct_hadiths",
    "title": "آيات خاطئة وأحاديث صحيحة",
    "question": "ما الدليل على فضل الطهارة والنية من القرآن والسنة؟",
    "scenario": "سيناريو للتجربة: آيات خاطئة وأحاديث صحيحة",
    "text": "قال الله تعالى: \"إِنَّ اللَّهَ يُحِبُّ التَّوَّابِينَ وَيُحِبُّ الْمُسْرِفِينَ\". وقال تعالى: \"وَثِيَابَكَ فَطَهِّرْ وَالرُّجْزَ فَاعْبُدْ\". وأما السنة فقال رسول الله ﷺ: \"الطُّهُورُ شَطْرُ الْإِيمَانِ\" رواه مسلم. وقال ﷺ: \"إِنَّمَا الْأَعْمَالُ بِالنِّيَّاتِ ، وَإِنَّمَا لِكُلِّ امْرِئٍ مَا نَوَى\" رواه البخاري.",
    "built_from": "كُتب من نصوص القرآن والحديث المضمّنة؛ الأخطاء مصطنعة لغرض التجربة",
    "observed_statuses": [
      "CORRECTED",
      "CORRECTED",
      "VERIFIED",
      "VERIFIED"
    ]
  },
  {
    "id": "scenario_correct_ayahs_wrong_hadiths",
    "title": "آيات صحيحة وأحاديث خاطئة",
    "question": "ما الدليل على فضل الصدق وبر الوالدين والصلاة؟",
    "scenario": "سيناريو للتجربة: آيات صحيحة وأحاديث خاطئة",
    "text": "قال الله تعالى: \"يَا أَيُّهَا الَّذِينَ آمَنُوا اتَّقُوا اللَّهَ وَكُونُوا مَعَ الصَّادِقِينَ\" [التوبة: 119]. وقال تعالى: \"وَقَضَىٰ رَبُّكَ أَلَّا تَعْبُدُوا إِلَّا إِيَّاهُ وَبِالْوَالِدَيْنِ إِحْسَانًا\" [الإسراء: 23]. وقال رسول الله ﷺ: \"مَنْ سَافَرَ يَوْمَ الْأَرْبِعَاءِ لَمْ يَرْجِعْ إِلَّا بِالْخَسَارَةِ وَالْهَمِّ\". وقال ﷺ: \"مَنْ دَخَلَ بَيْتَهُ بِرِجْلِهِ الْيُسْرَى انْتَقَصَ عُمُرُهُ سَنَةً كَامِلَةً\".",
    "built_from": "كُتب من نصوص القرآن والحديث المضمّنة؛ الأخطاء مصطنعة لغرض التجربة",
    "observed_statuses": [
      "VERIFIED",
      "VERIFIED",
      "UNSUPPORTED",
      "UNSUPPORTED"
    ]
  },
  {
    "id": "scenario_all_correct_mixed",
    "title": "آيات صحيحة وأحاديث صحيحة",
    "question": "ما فضل الصدق والصبر والنية؟",
    "scenario": "سيناريو للتجربة: آيات صحيحة وأحاديث صحيحة",
    "text": "قال الله تعالى: \"يَا أَيُّهَا الَّذِينَ آمَنُوا اتَّقُوا اللَّهَ وَكُونُوا مَعَ الصَّادِقِينَ\" [التوبة: 119]. وقال رسول الله ﷺ: \"الطُّهُورُ شَطْرُ الْإِيمَانِ\" رواه مسلم. وقال تعالى: \"وَاسْتَعِينُوا بِالصَّبْرِ وَالصَّلَاةِ ۚ وَإِنَّهَا لَكَبِيرَةٌ إِلَّا عَلَى الْخَاشِعِينَ\" [البقرة: 45]. وقال ﷺ: \"إِنَّمَا الْأَعْمَالُ بِالنِّيَّاتِ ، وَإِنَّمَا لِكُلِّ امْرِئٍ مَا نَوَى\" رواه البخاري.",
    "built_from": "كُتب من نصوص القرآن والحديث المضمّنة؛ الأخطاء مصطنعة لغرض التجربة",
    "observed_statuses": [
      "VERIFIED",
      "VERIFIED",
      "VERIFIED",
      "VERIFIED"
    ]
  },
  {
    "id": "scenario_all_wrong_mixed",
    "title": "آيات خاطئة وأحاديث خاطئة",
    "question": "ما الآيات والأحاديث الواردة في الرزق والعمل؟",
    "scenario": "سيناريو للتجربة: آيات خاطئة وأحاديث خاطئة",
    "text": "قال الله تعالى: \"وَمَا خَلَقْتُ الْجِنَّ وَالْإِنسَ إِلَّا لِيَأْكُلُونِ\". وقال ﷺ: \"مَنْ لَبِسَ الْأَخْضَرَ يَوْمَ الْجُمُعَةِ زَادَ اللَّهُ فِي رِزْقِهِ عَشْرَةَ أَضْعَافٍ\". وقال تعالى: \"فَإِنَّ مَعَ الْعُسْرِ يُسْرًا ۝ إِنَّ بَعْدَ الْعُسْرِ سُرُورًا\". وقال ﷺ: \"مَنْ قَالَ سُبْحَانَ اللَّهِ مِائَةَ مَرَّةٍ دَخَلَ الْجَنَّةَ بِلَا حِسَابٍ وَلَا عَذَابٍ\".",
    "built_from": "كُتب من نصوص القرآن والحديث المضمّنة؛ الأخطاء مصطنعة لغرض التجربة",
    "observed_statuses": [
      "CORRECTED",
      "UNSUPPORTED",
      "CORRECTED",
      "HUMAN_REVIEW"
    ]
  },
  {
    "id": "correct_ayah",
    "title": "آية صحيحة",
    "scenario": "correct",
    "text": "قال الله تعالى: \"يَا أَيُّهَا الَّذِينَ آمَنُوا اسْتَعِينُوا بِالصَّبْرِ وَالصَّلَاةِ ۚ إِنَّ اللَّهَ مَعَ الصَّابِرِينَ\". وفي الآية بشارة لمن صبر.",
    "built_from": [
      "سورة البقرة 153"
    ],
    "observed_statuses": [
      "VERIFIED"
    ]
  },
  {
    "id": "correct_hadith",
    "title": "حديث صحيح",
    "scenario": "correct",
    "text": "وقال رسول الله ﷺ: \"إِنَّمَا الْأَعْمَالُ بِالنِّيَّاتِ ، وَإِنَّمَا لِكُلِّ امْرِئٍ مَا نَوَى ، فَمَنْ كَانَتْ هِجْرَتُهُ إِلَى دُنْيَا يُصِيبُهَا ، أَوْ إِلَى امْرَأَةٍ يَنْكِحُهَا ، فَهِجْرَتُهُ إِلَى مَا هَاجَرَ إِلَيْهِ\". ويستفاد منه أهمية النية.",
    "built_from": [
      "حديث رقم 5"
    ],
    "observed_statuses": [
      "VERIFIED"
    ]
  },
  {
    "id": "substituted_word",
    "title": "كلمة مستبدلة: «ثم استقم» بدل «فاستقم»",
    "scenario": "altered",
    "text": "قال الله تعالى: \"ثُمَّ اسْتَقِمْ كَمَا أُمِرْتَ وَمَنْ تَابَ مَعَكَ\".",
    "built_from": [
      "سورة هود 112 (فَاسْتَقِمْ ← ثُمَّ اسْتَقِمْ)"
    ],
    "observed_statuses": [
      "CORRECTED"
    ]
  },
  {
    "id": "swapped_order",
    "title": "تبديل ترتيب الكلمات",
    "scenario": "altered",
    "text": "قال الله تعالى: \"كَمَا أُمِرْتَ فَاسْتَقِمْ وَمَنْ تَابَ مَعَكَ\".",
    "built_from": [
      "سورة هود 112 (تقديم وتأخير)"
    ],
    "observed_statuses": [
      "CORRECTED"
    ]
  },
  {
    "id": "missing_diacritics",
    "title": "بدون تشكيل (مقبول)",
    "scenario": "correct",
    "text": "قال الله تعالى: \"فاستقم كما أمرت ومن تاب معك\".",
    "built_from": [
      "سورة هود 112 بلا تشكيل"
    ],
    "observed_statuses": [
      "VERIFIED"
    ]
  },
  {
    "id": "partial_citation",
    "title": "اقتباس جزئي من آية",
    "scenario": "partial",
    "text": "قال الله تعالى: \"۞ وَقَضَىٰ رَبُّكَ أَلَّا تَعْبُدُوا إِلَّا\".",
    "built_from": [
      "سورة الإسراء 23 (أول ست كلمات)"
    ],
    "observed_statuses": [
      "VERIFIED"
    ]
  },
  {
    "id": "missing_word",
    "title": "كلمة ناقصة",
    "scenario": "altered",
    "text": "قال الله تعالى: \"يَا أَيُّهَا آمَنُوا اسْتَعِينُوا بِالصَّبْرِ وَالصَّلَاةِ ۚ إِنَّ اللَّهَ مَعَ الصَّابِرِينَ\".",
    "built_from": [
      "سورة البقرة 153 (حُذفت الكلمة الثالثة)"
    ],
    "observed_statuses": [
      "CORRECTED"
    ]
  },
  {
    "id": "unannounced_quote",
    "title": "اقتباس غير معلَن بلا علامات تنصيص",
    "scenario": "unannounced",
    "text": "وهنا نتذكر أن العبد عليه أن يثبت وألا يتراجع، فاستقم كما أمرت ومن تاب معك ولا تطغوا، وهذا من أعظم ما يعين على الطاعة.",
    "built_from": [
      "سورة هود 112 داخل الجملة"
    ],
    "observed_statuses": [
      "VERIFIED"
    ]
  },
  {
    "id": "uncertain_hadith",
    "title": "حديث مركّب من حديثين",
    "scenario": "uncertain",
    "text": "روى البخاري أن النبي ﷺ قال: \"إِنَّمَا الْأَعْمَالُ بِالنِّيَّاتِ ، وَإِنَّمَا لِكُلِّ امْرِئٍ مَا نَوَى ، فَمَنْ كَانَتْ هِجْرَتُهُ أَهْلَ الْأَرْضِ يَرْحَمْكُمْ مَنْ فِي السَّمَاءِ\".",
    "built_from": [
      "حديث رقم 5 (النصف الأول)",
      "حديث رقم 95766 (النصف الثاني)"
    ],
    "observed_statuses": [
      "HUMAN_REVIEW"
    ],
    "question": "ما الحديث الوارد في النية والهجرة؟"
  },
  {
    "id": "no_quotation",
    "title": "نص بلا اقتباسات",
    "scenario": "none",
    "text": "الصبر من أهم الصفات التي ينبغي أن يتحلى بها الإنسان في حياته، ويساعده على مواجهة المصاعب بثبات وهدوء.",
    "built_from": [],
    "observed_statuses": []
  },
  {
    "id": "mixed_response",
    "title": "رد مختلط",
    "scenario": "mixed",
    "text": "بخصوص بر الوالدين، قال الله تعالى: \"۞ وَقَضَىٰ رَبُّكَ أَلَّا تَعْبُدُوا إِلَّا إِيَّاهُ وَبِالْوَالِدَيْنِ إِحْسَانًا ۚ إِمَّا يَبْلُغَنَّ عِنْدَكَ الْكِبَرَ أَحَدُهُمَا أَوْ كِلَاهُمَا فَلَا تَقُلْ لَهُمَا أُفٍّ وَلَا تَنْهَرْهُمَا وَقُلْ لَهُمَا قَوْلًا كَرِيمًا\". وقال رسول الله ﷺ: \"الرَّاحِمُونَ يَرْحَمُهُمُ الرَّحْمَنُ ، ارْحَمُوا أَهْلَ الْأَرْضِ يَرْحَمْكُمْ مَنْ فِي السَّمَاءِ\". وقال سبحانه: \"يَا أَيُّهَا آمَنُوا اسْتَعِينُوا بِالصَّبْرِ وَالصَّلَاةِ ۚ إِنَّ اللَّهَ مَعَ الصَّابِرِينَ\". وهذا من مكارم الأخلاق.",
    "built_from": [
      "سورة الإسراء 23",
      "حديث رقم 95766",
      "سورة البقرة 153 (كلمة ناقصة)"
    ],
    "observed_statuses": [
      "VERIFIED",
      "VERIFIED",
      "CORRECTED"
    ],
    "question": "ما حكم بر الوالدين وما الدليل عليه؟"
  },
  {
    "id": "extra_short_quote_review",
    "title": "اقتباس قصير جدًا → مراجعة بشرية",
    "question": "ما الحديث الذي ورد في فضل الرحمة؟",
    "scenario": "اقتباس قصير جدًا → مراجعة بشرية",
    "text": "ورد في فضل الرحمة قوله ﷺ: \"...الراحم عطية\" رواه أبو داود.",
    "built_from": "نصوص القرآن والحديث المضمّنة؛ الأخطاء مصطنعة لغرض التجربة",
    "observed_statuses": [
      "HUMAN_REVIEW"
    ]
  },
  {
    "id": "extra_hadith_one_word_changed",
    "title": "حديث تغيّرت فيه كلمة → مراجعة بشرية",
    "question": "ماذا قال النبي ﷺ عن الكلام والصمت؟",
    "scenario": "حديث تغيّرت فيه كلمة → مراجعة بشرية",
    "text": "قال رسول الله ﷺ: \"مَنْ كَانَ يُؤْمِنُ بِاللَّهِ وَالْيَوْمِ الْآخِرِ فَلْيَقُلْ شَرًّا أَوْ لِيَصْمُتْ\" متفق عليه.",
    "built_from": "نصوص القرآن والحديث المضمّنة؛ الأخطاء مصطنعة لغرض التجربة",
    "observed_statuses": [
      "HUMAN_REVIEW"
    ]
  },
  {
    "id": "extra_hadith_attributed_to_god",
    "title": "حديث نُسب إلى الله تعالى",
    "question": "هل في القرآن أن الطهور شطر الإيمان؟",
    "scenario": "حديث نُسب إلى الله تعالى",
    "text": "جاء في القرآن الكريم: قال الله تعالى: \"الطُّهُورُ شَطْرُ الْإِيمَانِ\".",
    "built_from": "نصوص القرآن والحديث المضمّنة؛ الأخطاء مصطنعة لغرض التجربة",
    "observed_statuses": [
      "UNSUPPORTED"
    ]
  },
  {
    "id": "extra_ayah_attributed_to_prophet",
    "title": "آية نُسبت إلى النبي ﷺ",
    "question": "ما الحديث الذي فيه أن مع العسر يسرا؟",
    "scenario": "آية نُسبت إلى النبي ﷺ",
    "text": "قال رسول الله ﷺ: \"فَإِنَّ مَعَ الْعُسْرِ يُسْرًا ۝ إِنَّ مَعَ الْعُسْرِ يُسْرًا\".",
    "built_from": "نصوص القرآن والحديث المضمّنة؛ الأخطاء مصطنعة لغرض التجربة",
    "observed_statuses": [
      "VERIFIED"
    ]
  },
  {
    "id": "extra_uncertain_ayah",
    "title": "آية غير مؤكدة → مراجعة بشرية",
    "question": "اذكر آية عن التقوى والرزق.",
    "scenario": "آية غير مؤكدة → مراجعة بشرية",
    "text": "قال الله تعالى: \"وَمَنْ يَتَّقِ اللَّهَ يَجْعَلْ لَهُ مِنْ كُلِّ ضِيقٍ مَخْرَجًا وَيَرْزُقْهُ مِنْ حَيْثُ لَا يَشْعُرُ بِالنَّجَاحِ\".",
    "built_from": "نصوص القرآن والحديث المضمّنة؛ الأخطاء مصطنعة لغرض التجربة",
    "observed_statuses": [
      "HUMAN_REVIEW"
    ]
  },
  {
    "id": "extra_mixed_all_outcomes",
    "title": "نتائج متنوعة في إجابة واحدة",
    "question": "ما فضل الصبر والتوكل والصدق؟",
    "scenario": "نتائج متنوعة في إجابة واحدة",
    "text": "قال الله تعالى: \"إِنَّ اللَّهَ مَعَ الصَّابِرِينَ\" [البقرة: 153]. وقال تعالى: \"وَمَنْ يَتَوَكَّلْ عَلَى اللَّهِ فَهُوَ كَافِيهِ\" [الطلاق: 3]. وقال ﷺ: \"الطُّهُورُ شَطْرُ الْإِيمَانِ\" رواه مسلم. وقال ﷺ: \"مَنْ لَبِسَ الْأَخْضَرَ يَوْمَ الْجُمُعَةِ زَادَ اللَّهُ فِي رِزْقِهِ عَشْرَةَ أَضْعَافٍ\".",
    "built_from": "نصوص القرآن والحديث المضمّنة؛ الأخطاء مصطنعة لغرض التجربة",
    "observed_statuses": [
      "VERIFIED",
      "CORRECTED",
      "VERIFIED",
      "UNSUPPORTED"
    ]
  }
]

Loads the saved examples and prints the decision of every quotation in each one.

In [ ]:
examples = json.loads((WORKDIR / "demo" / "examples.json").read_text(encoding="utf-8"))
for example in examples:
    print("\n==", example["id"], "-", example["title"])
    table(pipeline.analyze(example["text"]))

## 8. Optional: Ask & Verify
Asks an LLM a question and verifies every quotation in its answer. Disabled unless `USE_LLM = True` and a key is stored in Colab *Secrets* as `OPENAI_API_KEY` (never paste a key into the notebook).

In [ ]:
if USE_LLM:
    from google.colab import userdata
    from llm_client import LLMSettings, generate, sanitize_answer
    settings = LLMSettings(api_key=userdata.get("OPENAI_API_KEY"), model=os.environ.get("OPENAI_MODEL", ""))
    question = "ما فضل الصبر في القرآن والسنة؟"
    answer = sanitize_answer(generate(settings, question))
    print(answer)
    verify(answer)
else:
    print("USE_LLM is False: skipping the optional LLM step.")

## 9. Export the official IslamicEval formats
Writes the three Subtask-1 submission files (1A spans, 1B labels, 1C corrections) for a dictionary of `{question_id: text}`.

In [ ]:
from benchmark_format import run_on_texts

texts = {"Q1": sample, **{e["id"]: e["text"] for e in examples[:3]}}
run_on_texts(pipeline, texts, out_dir="outputs")
for name in sorted(p.name for p in pathlib.Path("outputs").glob("*.tsv")):
    print("\n#", name); print(pathlib.Path("outputs", name).read_text(encoding="utf-8")[:400])

## 10. Optional stress test
Samples real ayahs and Hadith from the corpora, builds exact, re-spelled, partial, altered and fabricated variants and reports detection, verdict and correction rates per kind.

In [ ]:
%%writefile stress_test.py

from __future__ import annotations

import random
import re
import sys
from collections import defaultdict
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from normalization import normalize_for_matching  # noqa: E402
from verifier import IslamicContentVerifier  # noqa: E402

DIAC = re.compile("[ؐ-ًؚ-ٰٟۖ-ۭـ]")
INTROS_Q = ["قال الله تعالى: ", "وفي القرآن الكريم: ", "ويقول ربنا سبحانه: ", "جاء في كتاب الله: "]
INTROS_H = ["قال رسول الله ﷺ: ", "وفي الحديث الشريف: ", "روى البخاري أن النبي ﷺ قال: ", "وقال النبي ﷺ: "]
TAIL = ["، وهذا واضح.", ". وبالله التوفيق.", "، فتأمل ذلك."]


def strip(text: str) -> str:
    return re.sub(r"\s+", " ", DIAC.sub("", text)).strip()


def build_cases(kb, n: int, rng: random.Random):
    cases = []
    quran = [r for r in kb.quran if len(r["text"].split()) >= 4]
    vocab = sorted({w for r in kb.quran for w in strip(r["text"]).split()})
    hadith = [r for r in kb.hadith if r["matn"] and len(r["matn"].split()) >= 8]

    def wrap(kind, label, quote, expect, source=None, mode="quoted", gold=None):
        intro = rng.choice(INTROS_Q if label == "Ayah" else INTROS_H)
        if mode == "quoted":
            text, q = f"{intro}\"{quote}\"{rng.choice(TAIL)}", quote
        elif mode == "bare":
            text, q = f"{intro}{quote}{rng.choice(TAIL)}", quote
        else:  
            text, q = f"ونختم هذا الحديث بما يلي {quote} والله أعلم.", quote
        start = text.index(q)
        cases.append({"kind": kind, "label": label, "text": text, "span": (start, start + len(q)), "expect": expect,
                      "gold": gold, "mode": mode})

    for rec in rng.sample(quran, n):
        words = rec["text"].split()
        plain = strip(rec["text"]).split()
        wrap("quran_exact", "Ayah", rec["text"], "VERIFIED")
        wrap("quran_plain", "Ayah", " ".join(plain), "VERIFIED")
        wrap("quran_unmarked", "Ayah", " ".join(plain), "VERIFIED", mode="bare")
        wrap("quran_embedded", "Ayah", " ".join(plain), "VERIFIED", mode="embedded")
        if len(plain) >= 8:
            a = rng.randint(0, len(plain) - 6)
            wrap("quran_partial", "Ayah", " ".join(plain[a:a + rng.randint(5, min(8, len(plain) - a))]), "VERIFIED")
        if len(plain) >= 5:
            i = rng.randrange(1, len(plain) - 1)
            alt = plain[:i] + [rng.choice([w for w in vocab if w != plain[i]])] + plain[i + 1:]
            wrap("quran_substituted", "Ayah", " ".join(alt), "CORRECTED", gold=" ".join(plain))
            alt = plain[:i] + plain[i + 1:]
            wrap("quran_dropped", "Ayah", " ".join(alt), "CORRECTED", gold=" ".join(plain))
            j = rng.randrange(0, len(plain) - 1)
            sw = list(plain); sw[j], sw[j + 1] = sw[j + 1], sw[j]
            if sw != plain:
                wrap("quran_swapped", "Ayah", " ".join(sw), "CORRECTED", gold=" ".join(plain))
        fake = [rng.choice(vocab) for _ in range(rng.randint(6, 10))]
        wrap("quran_fabricated", "Ayah", " ".join(fake), "NOT_VERIFIED")
    for rec in rng.sample(hadith, n):
        words = strip(rec["matn"]).split()
        k = rng.randint(7, min(24, len(words)))
        a = rng.randint(0, len(words) - k)
        frag = words[a:a + k]
        raw = rec["matn"].split()
        rawfrag = raw[a:a + k] if len(raw) == len(words) else frag
        wrap("hadith_exact", "Hadith", " ".join(rawfrag), "VERIFIED")
        wrap("hadith_plain", "Hadith", " ".join(frag), "VERIFIED")
        wrap("hadith_unmarked", "Hadith", " ".join(frag), "VERIFIED", mode="bare")
        other = rng.choice(hadith)
        ow = strip(other["matn"]).split()
        half = frag[: k // 2] + ow[: max(4, k // 2)]
        wrap("hadith_spliced", "Hadith", " ".join(half), "NOT_VERIFIED")
        fake = [rng.choice(vocab) for _ in range(rng.randint(8, 12))]
        wrap("hadith_fabricated", "Hadith", " ".join(fake), "NOT_VERIFIED")
    return cases


def overlap(a, b):
    return max(0, min(a[1], b[1]) - max(a[0], b[0])) / max(1, b[1] - b[0])


def evaluate(cases, pipe, show=15):
    stats = defaultdict(lambda: defaultdict(int))
    failures = defaultdict(list)
    for c in cases:
        result = pipe.analyze(c["text"])
        best = max(result["spans"], key=lambda s: overlap((s["start"], s["end"]), c["span"]), default=None)
        s = stats[c["kind"]]
        s["n"] += 1
        found = best is not None and overlap((best["start"], best["end"]), c["span"]) >= 0.6
        fabricated = c["expect"] == "NOT_VERIFIED"
        if found:
            s["found"] += 1
            s["typed"] += best["type"] == c["label"]
        status = best["status"] if found else "MISSED"
        if fabricated:
            ok = status != "VERIFIED"      
            s["ok"] += ok
            s["false_verified"] += status == "VERIFIED"
        elif c["expect"] == "VERIFIED":
            ok = status == "VERIFIED"
            s["ok"] += ok
        else:
            ok = status in ("CORRECTED",)
            s["ok"] += ok
            s["flagged"] += status in ("CORRECTED", "HUMAN_REVIEW", "UNSUPPORTED")
            if status == "CORRECTED":
                fixed = strip(best["correction"]["display_text"]) if best["correction"] else ""
                s["fix_exact"] += normalize_for_matching(fixed) == normalize_for_matching(c["gold"])
        if not ok and len(failures[c["kind"]]) < show:
            failures[c["kind"]].append((status, c["text"][:150]))
    return stats, failures


def main(argv):
    n = int(argv[1]) if len(argv) > 1 else 60
    seed = int(argv[2]) if len(argv) > 2 else 7
    pipe = IslamicContentVerifier()
    cases = build_cases(pipe.retriever, n, random.Random(seed))
    stats, failures = evaluate(cases, pipe)
    print(f"{'kind':20s} {'n':>4s} {'found':>6s} {'typed':>6s} {'ok':>6s} extra")
    for kind, s in sorted(stats.items()):
        extra = ""
        if "fix_exact" in s: extra = f"flagged={s['flagged']/s['n']:.2f} fix_exact={s['fix_exact']/s['n']:.2f}"
        if "false_verified" in s: extra = f"false_verified={s['false_verified']}"
        print(f"{kind:20s} {s['n']:4d} {s['found']/s['n']:6.2f} {s['typed']/max(1,s['found']):6.2f} {s['ok']/s['n']:6.2f} {extra}")
    if "--fail" in argv:
        for kind, rows in failures.items():
            print("\n==", kind)
            for status, text in rows:
                print(" ", status, "|", text)


if __name__ == "__main__":
    main(sys.argv)

Runs 20 sampled cases per kind (seed 7) and prints detection, verdict and correction rates; any fabricated text marked verified would be listed.

In [ ]:
import random, importlib, stress_test
importlib.reload(stress_test)
cases = stress_test.build_cases(pipeline.retriever, 20, random.Random(7))
stats, failures = stress_test.evaluate(cases, pipeline)
for kind, s in sorted(stats.items()):
    print(f"{kind:20s} n={s['n']:3d} found={s['found']/s['n']:.2f} ok={s['ok']/s['n']:.2f} false_verified={s.get('false_verified', 0)}")

## 11. Anti-hallucination self-check
Every reference and correction in a report must be found word for word inside the cited record. The check below fails loudly otherwise.

In [ ]:
checked = 0
for example in examples:
    for span in pipeline.analyze(example["text"])["spans"]:
        assert span["grounding"] == "verified", (example["id"], span["status"])
        checked += 1
print(f"All {checked} reports are grounded in the bundled corpora.")